##  Header

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import warnings
import matplotlib_inline
import seaborn as sns
from scipy.special import jv
import glob
import os
import re
# import xarray_sf_funcs as xsfuncs
import importlib
# importlib.reload(xsfuncs)
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature

warnings.filterwarnings("ignore")

sns.set_style(style="white")
sns.set_context("notebook")

# plt.rcParams["figure.figsize"] = [4,3]
# plt.rcParams['figure.dpi'] = 100

matplotlib_inline.backend_inline.set_matplotlib_formats("retina")

plt.rcParams["text.usetex"] =False
plt.rcParams["xtick.bottom"] = True
plt.rcParams["ytick.left"] = True
plt.rcParams["xtick.top"] = True
plt.rcParams["ytick.right"] = True

# use latex fonts for math text in plots
plt.rcParams["mathtext.fontset"] = "stix"

# force ticks to go inward for all plots
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

### Funcs

In [2]:
def _format_date_range(ds, time_dim="time"):

    start = ds[time_dim].min(skipna=True).dt.strftime("%b %d, %Y").item()
    end = ds[time_dim].max(skipna=True).dt.strftime("%b %d, %Y").item()

    # print(f"Formatting date range for dataset with time dimension of shape {ds.time.shape}")
    # print(f"Dataset time range: {start} to {end}")

    return f"{start} – {end}"

# Function to convert wavenumber to separation distance
def wavenumber_to_distance(wavenumber):
    return 1 / wavenumber

# Function to convert separation distance to wavenumber
def distance_to_wavenumber(distance):
    return 1 / distance

# Function to process datasets
def process_dataset(ds, KEflux_key, QFlux_key, quantile_upper, quantile_lower, dim):
    if ds is not None:
        ds = ds.where(
            (ds[KEflux_key] < ds[KEflux_key].quantile(quantile_upper, dim))
            & (ds[KEflux_key] > ds[KEflux_key].quantile(quantile_lower, dim))
            # & (ds[QFlux_key] < ds[QFlux_key].quantile(quantile_upper, dim))
            # & (ds[QFlux_key] > ds[QFlux_key].quantile(quantile_lower, dim))
        )
        ds["EFlux_CG"] = ds["EFlux_CG"].mean(dim="shiftnum")
        ds["K_coarse_grain"] = ds["K_coarse_grain"].mean(dim="shiftnum")

        try:
            ds["QFlux_CG"] = ds["QFlux_CG"].mean(dim="shiftnum")
        except KeyError:
            pass
    return ds

def load_dataset(file_key, params, region, preprocess_func=None):
    try:
        file_path = params[file_key]
        if not file_path.endswith(".nc"):
            raise ValueError(f"Invalid file for {file_key} in region {region}: {file_path}")
        ds = xr.open_dataset(file_path)
        if preprocess_func:
            ds = preprocess_func(ds)
            # ds = ds.load()  # Load into memory to close netCDF handles immediately, then convert to dask arrays for lazy operations
            # ds = ds.chunk()
        return ds
    except (ValueError, FileNotFoundError, KeyError) as e:
        print(f"Skipping {file_key} dataset for region {region} due to error: {e}")
        return None

def preprocess_LLC4320(ds, region):
    # ds_og = xr.open_mfdataset(
    #     f'/Volumes/Promise Disk/data/mitgcm/{region}_data/*.nc',
    #     combine="nested",
    #     concat_dim="time",
    #     coords="minimal",
    #     compat="override",
    # )
    ds['asf_flux_mean'] = -0.5 * ds['asf_mean'] * (1318 / 2000) ** 2
    # ds = ds.assign_coords({'time': ds_og['time']})
    return ds

def preprocess_NEMO(ds, mindate=None, maxdate=None, snapshot_frequency=1, og_filename='/Volumes/Promise Disk/data/NEMO/2019-2021_data/cmems_mod_glo_phy_my_0.083deg_P1D-m_uo-vo_180.00W-179.92E_80.00S-90.00N_0.49m_2019-06-30-2021-06-30.nc'):

    ds_og = xr.open_dataset(og_filename)
    ds = ds.assign_coords({'time': ds_og['time']})
    ds = ds.sel(time=slice(mindate, maxdate, snapshot_frequency))
    ds['asf_flux_mean'] = -0.5 * ds['asf_mean']
    return ds

# Function to plot a dataset
def plot_dataset(ax, ax_sec, ds, dim, KEflux_key, dwAw_key, LLL_key, Lww_key, x_key, scale_factor, quantile_alpha, quantile_upper, quantile_lower, confint, SF, CG, Ens, KE, dwAw, LLL, Ens_LLL, Lww, ke_flux_label, enstrophy_flux_label, dwdAw_label, CG_label, CG_label_q, LLL_label, Ens_LLL_label, Lww_label, convert_KEFlux_to_wattperkm2permeter, convert_KEFlux_to_Wperm3, colors):

    blue, lightblue, red, lightred = colors
    if Ens and not KE:
        ax_sec = ax

    if ds is not None:

        if convert_KEFlux_to_wattperkm2permeter:
            # Convert from m^2/s^3 to W/km^2/m assuming density = 1025 kg/m^3
            conversion_scale_factor = 1025 / 1e-6
        if convert_KEFlux_to_Wperm3:
            conversion_scale_factor = 1025
        else:
            conversion_scale_factor = 1.0

        if SF:

            if KE: 
                ax.semilogx(2 * np.pi / ds[x_key],
                    -0.5 * ds[KEflux_key].mean(dim) * scale_factor * conversion_scale_factor,
                    color=blue, zorder=1, label='Structure function: ' + ke_flux_label)
                
                if LLL:
                    ax.semilogx(2 * np.pi / ds[x_key],
                        -2 * ds[LLL_key].mean(dim) * scale_factor * conversion_scale_factor / (3 * ds[x_key]),
                        color=blue, linestyle="--", zorder=1, label='Structure function: ' + LLL_label)

            if Ens:
                ax_sec.semilogx(2 * np.pi / ds[x_key],
                    2 * ds[KEflux_key].mean(dim) * scale_factor / ds[x_key]**2,
                    color=red, zorder=1, label='Structure function: ' + enstrophy_flux_label)
                
                if Lww:
                    ax_sec.semilogx(2 * np.pi / ds[x_key],
                        -0.5 * ds[Lww_key].mean(dim) * scale_factor / ds[x_key],
                        color=red, linestyle="--", zorder=1, label='Structure function: ' + Lww_label)
                
                if dwAw:
                    ax_sec.semilogx(2 * np.pi / ds[x_key],
                        -0.5 * ds[dwAw_key].mean(dim) * scale_factor,
                        color=red, linestyle=":", zorder=1, label='Structure function: ' + dwdAw_label)
                
                if Ens_LLL:
                    ax_sec.semilogx(2 * np.pi / ds[x_key],
                        8 * ds[LLL_key].mean(dim) * scale_factor / ds[x_key]**3,
                        color=blue, linestyle=(0, (3, 5, 1, 5)), zorder=1, label='Structure function: ' + Ens_LLL_label)
            
            if confint:
                if KE:
                    ax.fill_between(2 * np.pi / ds[x_key],
                        -0.5 * ds[KEflux_key].quantile(quantile_lower, dim=dim) * scale_factor * conversion_scale_factor,
                        -0.5 * ds[KEflux_key].quantile(quantile_upper, dim=dim) * scale_factor * conversion_scale_factor,
                        color=blue, alpha=quantile_alpha)
                    
                    if LLL:
                        ax.fill_between(2 * np.pi / ds[x_key],
                            -2 * ds[LLL_key].quantile(quantile_lower, dim=dim) * scale_factor * conversion_scale_factor / (3 * ds[x_key]),
                            -2 * ds[LLL_key].quantile(quantile_upper, dim=dim) * scale_factor * conversion_scale_factor / (3 * ds[x_key]),
                            color=blue, alpha=quantile_alpha)
                
                if Ens:
                    ax_sec.fill_between(2 * np.pi / ds[x_key],
                        2 * ds[KEflux_key].quantile(quantile_lower, dim=dim) * scale_factor / ds[x_key]**2,
                        2 * ds[KEflux_key].quantile(quantile_upper, dim=dim) * scale_factor / ds[x_key]**2,
                        color=red, alpha=quantile_alpha)
                    
                    if dwAw:
                        ax_sec.fill_between(2 * np.pi / ds[x_key],
                            -0.5 * ds[dwAw_key].quantile(quantile_lower, dim=dim) * scale_factor,
                            -0.5 * ds[dwAw_key].quantile(quantile_upper, dim=dim) * scale_factor,
                            color=red, alpha=quantile_alpha)
                        
                    if Lww:
                        ax_sec.fill_between(2 * np.pi / ds[x_key],
                            -0.5 * ds[Lww_key].quantile(quantile_lower, dim=dim) * scale_factor / ds[x_key],
                            -0.5 * ds[Lww_key].quantile(quantile_upper, dim=dim) * scale_factor / ds[x_key],
                            color=red, alpha=quantile_alpha)
                        
                    if Ens_LLL:
                        ax_sec.fill_between(2 * np.pi / ds[x_key],
                            8 * ds[LLL_key].quantile(quantile_lower, dim=dim) * scale_factor / ds[x_key]**3,
                            8 * ds[LLL_key].quantile(quantile_upper, dim=dim) * scale_factor / ds[x_key]**3,
                            color=blue, alpha=quantile_alpha)

        if CG:

            if KE:
        
                ax.semilogx(ds["K_coarse_grain"].mean(dim),
                    ds["EFlux_CG"].mean(dim) * scale_factor * conversion_scale_factor,
                    color=lightblue, linestyle="-.", label=CG_label, zorder=0)
                
                if confint:
                    ax.fill_between(ds["K_coarse_grain"].mean(dim),
                        ds["EFlux_CG"].quantile(quantile_lower, dim=dim) * scale_factor * conversion_scale_factor,
                        ds["EFlux_CG"].quantile(quantile_upper, dim=dim) * scale_factor * conversion_scale_factor,
                        color=lightblue, alpha=quantile_alpha)

            if Ens:
                ax_sec.semilogx(ds["K_coarse_grain"].mean(dim),
                    ds["QFlux_CG"].mean(dim) * scale_factor,
                    color=lightred, linestyle="-.", label=CG_label_q, zorder=0)
                if confint:
                    ax_sec.fill_between(ds["K_coarse_grain"].mean(dim),
                        ds["QFlux_CG"].quantile(quantile_lower, dim=dim) * scale_factor,
                        ds["QFlux_CG"].quantile(quantile_upper, dim=dim) * scale_factor,
                        color=lightred, alpha=quantile_alpha)


def set_common_properties(axes, axes_sec, datasets, params, Ens, KE, ke_flux_label, enstrophy_flux_label, convert_KEFlux_to_wattperkm2permeter, convert_KEFlux_to_Wperm3, KE_lims_scale, Q_lims_scale, draw_Rossby_radius, colors, KE_ymin_override=None, KE_ymax_override=None, Q_ymin_override=None, Q_ymax_override=None, legend_outside=False):
    blue, lightblue, red, lightred = colors

    if Ens and KE:
        enums = enumerate(zip(axes, axes_sec, datasets.keys()))
        
    else:
        enums = enumerate(zip(axes, datasets.keys()))

    for i, vars in enums:
        if Ens and KE:
            ax, ax_sec, key = vars
        else:
            ax, key = vars
        ds = datasets[key]
        ax.set_xlabel("Wavenumber [m$^{-1}$]")
        ax.set_xlim(1e-6, 1e-2)  # Set x-limits for all axes

        if ds is not None:
            # Set ymin and ymax to be symmetric around zero
            asf_mean = abs(-0.5 * ds_SWOT["asf_down"].mean("swath_num")).mean()

            if convert_KEFlux_to_wattperkm2permeter:
                scale_factor = 1025 / 1e-6
            if convert_KEFlux_to_Wperm3:
                scale_factor = 1025
            else:
                scale_factor = 1.0

            asf_mean *= scale_factor

            ax.set_ylim(-asf_mean * KE_lims_scale, asf_mean * KE_lims_scale)

            ax.hlines(0, 1e-6, 1e-2, colors="k", lw=1, zorder=0)

            if draw_Rossby_radius:
                R = params["Rossby Radius"]  # units of kilometers
                if R is not None:
                    k_Rossby = distance_to_wavenumber(R * 1e3)  # convert to meters and then to wavenumber
                    # ax.axvline(k_Rossby, color="grey", linestyle="--", label="Rossby Radius", lw=1)
                    # ax.text(k_Rossby * 1.1, ax.get_ylim()[1] * 0.8, f"Ro {R} km", color="grey")

                    # Use fill between to shade R+/-
                    ax.fill_betweenx(
                        ax.get_ylim(),
                        k_Rossby * 0.5,
                        k_Rossby * 1.5,
                        color="grey",
                        alpha=0.3,
                    )
                    # put text in the lower middle of the shaded area
                    ax.text(k_Rossby, ax.get_ylim()[0] * 0.8, "Ro", color="k", ha="center")

            # Set y-label for primary y-axis
            if i == 0:  # Only set for the first axis
                if convert_KEFlux_to_wattperkm2permeter:
                    ke_flux_units = " W km$^{-2}$ m$^{-1}$"
                if convert_KEFlux_to_Wperm3:
                    ke_flux_units = " W m$^{-3}$"
                else:
                    ke_flux_units = " m$^2$ s$^{-3}$"
                
                enstrophy_flux_units = " s$^{-3}$"
                
                if KE and not Ens:
                    ax.set_ylabel(rf"KE flux [{ke_flux_units}]")
                elif not KE and Ens:
                    ax.set_ylabel(rf"Enstrophy flux [{enstrophy_flux_units}]")
            
            ax.tick_params(axis='x', direction="in", which="both", top=False)

            t = ax.yaxis.get_offset_text()
            t.set_x(-0.05)  # Adjust the position of the offset text

            # Add secondary x-axis for separation distance
            secax = ax.secondary_xaxis('top', functions=(wavenumber_to_distance, distance_to_wavenumber))
            secax.set_xlabel("Separation Distance [m]")
            secax.set_xlim(wavenumber_to_distance(1e-2), wavenumber_to_distance(1e-6))  # Match limits to primary x-axis
            secax.tick_params(direction="in", which="both", bottom=False)

            # Set properties for secondary y-axis
            if Ens:
                if KE:
                    if i == len(axes) - 1:  # Only set for the last axis
                        ax_sec.set_ylabel(rf"Enstrophy flux [{enstrophy_flux_units}]"
                                        f"\n {enstrophy_flux_label}",
                                        color=red)
                    ax.tick_params(axis='y', labelcolor=blue, direction="in")
                    ax_sec.tick_params(axis='y', labelcolor=red, direction="in")
                    t = ax_sec.yaxis.get_offset_text()
                    t.set_x(1.1)  # Adjust the position of the offset text

                if not KE:
                    ax_sec = ax

                enstrophy_mean = abs((2 * ds_SWOT["asf_down"].mean("swath_num") / ds_SWOT["num_lines_diffs"]**2)).mean()
                ax_sec.set_ylim(-enstrophy_mean * Q_lims_scale, enstrophy_mean * Q_lims_scale)

        # Add box with dataset name in the bottom left corner of each panel
        text = key if ds is not None else f"No {key} data"
        # text = 'SimSWOT no noise' if key == 'SimSWOT' else text
        # text = 'SimSWOT with noise' if key == 'SimSWOT_cleaned' else text
        ax.text(
            0.05, 0.05, text, transform=ax.transAxes,
            fontsize=10, verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8)
        )

    # Add box with full region name in the top left corner of the first axis
    axes[0].text(0.05, 0.95, params["Full name"], transform=axes[0].transAxes,
                 fontsize=10, verticalalignment='top', fontweight='bold', zorder=0,
                 bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))

    # Add legends
    axes[len(axes) - 1].legend(
        loc='upper left',  # Position the legend at the top center
        # # bbox_to_anchor=(0.5, 1.5),  # Adjust the position to be above the plot
        # ncol=2,  # Set the legend to have two columns
        # frameon=False  # Optional: Remove the legend box frame
        # # zorder=5
    )
    if legend_outside:
        axes[len(axes) - 1].legend(
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            frameon=False
        )

    if axes_sec is not None:
        axes_sec[len(axes_sec) - 1].legend(
            loc='upper right',  # Position the secondary legend at the upper center
            # bbox_to_anchor=(0.5, 1.2),  # Adjust the position to match the primary legend
            # ncol=2,  # Set the legend to have two columns
            # frameon=False  # Optional: Remove the legend box frame
            # # zorder=5
        )
        
        if legend_outside:
            axes_sec[len(axes_sec) - 1].legend(
                loc='center left',
                bbox_to_anchor=(1.02, 0.5),
                frameon=False
            )

    if KE_ymin_override is not None and KE_ymax_override is not None:
        if KE:
            for ax in axes:
                ax.set_ylim(KE_ymin_override, KE_ymax_override)
    if Q_ymin_override is not None and Q_ymax_override is not None:
        if Ens:
            if not KE:
                axes_sec = axes
            for ax_sec in axes_sec:
                ax_sec.set_ylim(Q_ymin_override, Q_ymax_override)

## Initialize region dictionary

### Funcs

In [3]:
def get_most_recent_file(region_name, file_pattern):
    """
    Get the most recent file for a specific region based on the datetime prefix in the filename.
    """
    # print(region_name, file_pattern)
    files = glob.glob(file_pattern)

    region_files = [f for f in files if ".nc" in os.path.basename(f)]
    if not region_files:
        raise FileNotFoundError(f"No files found for region: {region_name}")
    # Sort files by their datetime prefix (assuming the datetime is at the start of the filename)
    region_files.sort(key=lambda f: os.path.basename(f).split("_")[0], reverse=True)
    return region_files[0]

In [4]:
def merge_files_and_savenetcdf(region_output_dir, region, min_cycle=None, max_cycle=None, drop_cycles=None, cycle_list=None, startswith='20250730', date=False, mindate='', maxdate='',SWOT_name='SWOT_L3', suffix='CG'):

    if date and not (min_cycle or max_cycle):
        file_list = [f for f in os.listdir(region_output_dir) if f.endswith(".nc")]

        if not file_list:
            raise FileNotFoundError(f"No .nc files found in {region_output_dir} for region {region}.")
        datasets = [xr.open_dataset(os.path.join(region_output_dir, f)) for f in file_list if f.startswith(startswith) and region in f and re.search(rf"_\d+_{re.escape(suffix)}\.nc$", f)]
        if not datasets:
            raise FileNotFoundError(f"No datasets found for region {region} starting with {startswith} and suffix {suffix}.")
        merged_dataset = xr.concat(datasets, dim='time')
        output_file = os.path.join(region_output_dir, f"LLC4320_{region}_merged_dates_{mindate}-{maxdate}_{suffix}.nc")

        merged_dataset.to_netcdf(output_file, mode='w', format='NETCDF4', engine='netcdf4')

        print(f"Merged dataset saved to: {output_file}")


    if not date:
        if (min_cycle is None or max_cycle is None) and cycle_list is None:
            raise ValueError("Both min_cycle and max_cycle must be specified when date is False.")
        
        if cycle_list is not None:
            
            cycle_files = [f for f in os.listdir(region_output_dir) if SWOT_name in f and f.startswith(startswith) and f.endswith(".nc") and any(cycle in f for cycle in cycle_list)]
            if not cycle_files:
                raise FileNotFoundError(f"No cycle files found for region {region} with cycles {cycle_list}.")
        else:

            cycle_files = [f for f in os.listdir(region_output_dir) if SWOT_name in f and f.startswith(startswith) and f.endswith(".nc")]
            # Filter out cycles that are not in the specified range
            cycle_files = [f for f in cycle_files if re.match(rf".*{SWOT_name}_{region}_cycle_(\d+)_{suffix}\.nc", f) and min_cycle <= re.search(rf"{SWOT_name}_{region}_cycle_(\d+)_{suffix}\.nc", f).group(1) <= max_cycle]
            print(f"Cycle files for {SWOT_name} after filtering by min_cycle and max_cycle: {cycle_files}")

            if not cycle_files:
                # Try regex that looks for files formatted like "*_simSWOT_{region}_cycles_*-*_{suffix}.nc without specifying the cycle min and max
                cycle_files = [f for f in os.listdir(region_output_dir) if SWOT_name in f and f.startswith(startswith) and f.endswith(".nc") and re.match(rf".*_{SWOT_name}_{region}_cycles_\d+-\d+_{suffix}\.nc", f)]
                print(f"Cycle files: {cycle_files}")
        
        if not cycle_files:
            raise FileNotFoundError(f"No cycle files found for region {region} with cycles {min_cycle}-{max_cycle} or cycle_list {cycle_list}.")
        
        if drop_cycles is not None:
            cycle_files = [f for f in cycle_files if not any(drop_cycle in f for drop_cycle in drop_cycles)]
            if not cycle_files:
                raise FileNotFoundError(f"No cycle files left for region {region} after dropping cycles {drop_cycles}.")

        datasets = [xr.open_dataset(os.path.join(region_output_dir, f)) for f in cycle_files]

        try:
            merged_dataset = xr.concat(datasets, dim='swath_num')

        except ValueError:
            # Only keep variables that all to-be-merged datasets contain
            common_vars = set.intersection(*(set(ds.data_vars) for ds in datasets))
            if not common_vars:
                raise ValueError("No common variables found across datasets to concatenate.")

            try:
                filtered_datasets = [ds[sorted(common_vars)] for ds in datasets]
                merged_dataset = xr.concat(filtered_datasets, dim='swath_num')
                print(f"ValueError during concat; merged using only common variables: {sorted(common_vars)}")

            except xr.AlignmentError:
                # alignment error occurs if the datasets have different lengths along swath_num dimension
                # most likely only a couple datasets have this issue, so just drop those datasets and merge the rest
                print("AlignmentError during concat; trying to merge only datasets with matching swath_num lengths")
                swath_num_lengths = [ds.sizes['swath_num'] for ds in datasets]
                print(f"Swath_num lengths of datasets: {swath_num_lengths}")
                most_common_length = max(set(swath_num_lengths), key=swath_num_lengths.count)
                print(f"Most common swath_num length: {most_common_length}")
                filtered_datasets = [ds for ds in datasets if ds.sizes['swath_num'] == most_common_length]
                if not filtered_datasets:
                    raise ValueError("No datasets with matching swath_num lengths found to concatenate.")
                merged_dataset = xr.concat(filtered_datasets, dim='swath_num')
                print(f"AlignmentError during concat; merged using only datasets with swath_num length of {most_common_length}")

        # Add cycle number as a new coordinate
        # cycle_numbers = [int(re.search(rf"{SWOT_name}_{region}_cycle_(\d+)_CG\.nc", f).group(1)) for f in cycle_files]
        # merged_dataset = merged_dataset.assign_coords(cycle_num=("swath_num", cycle_numbers))
        
        # datetime_str = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

        if cycle_list is not None:
            output_file = os.path.join(region_output_dir, f"{SWOT_name}_{region}_cycles_{'-'.join(cycle_list)}_{suffix}.nc")
        else:
            output_file = os.path.join(region_output_dir, f"{SWOT_name}_{region}_cycles_{min_cycle}-{max_cycle}_{suffix}.nc")
        merged_dataset.to_netcdf(output_file, mode='w', format='NETCDF4', engine='netcdf4')
    
        print(f"Merged dataset saved to: {output_file}")

### Run pipeline

In [5]:
region_dict = {
    "acc": {
        "lat_north": -53,
        "lat_south": -57.5,
        "lon_east": 158,
        "lon_west": 148,
        "reduce_xd_num": 2,
        "LLC4320_file": "",
        "simSWOT_file": "",
        "NEMO_file": "",
        "Full name": "Antarctic Circumpolar Current",
        "Mean Latitude": " (55°S)",
        "Rossby Radius": 15,
    },
    "nwpacific": {
        "lat_north": 23,
        "lat_south": 19,
        "lon_east": 137,
        "lon_west": 132,
        "reduce_xd_num": 1,
        "LLC4320_file": "",
        "simSWOT_file": "",
        "NEMO_file": "",
        "Full name": "North Pacific",
        "Mean Latitude": " (21°N)",
        "Rossby Radius": 60,
    },
    "capebasin": {
        "lat_north": -41.01421,
        "lat_south": -44.99279,
        "lon_east": 16,
        "lon_west": 11,
        "reduce_xd_num": 1,
        "LLC4320_file": "",
        "simSWOT_file": "",
        "NEMO_file": "",
        "Full name": "Cape Basin",
        "Mean Latitude": " (43°S)",
        "Rossby Radius": 25,
    },
    "nwaustralia": {
        "lat_north": -11,
        "lat_south": -15,
        "lon_east": 125,
        "lon_west": 120,
        "reduce_xd_num": 1,
        "LLC4320_file": "",
        "simSWOT_file": "",
        "NEMO_file": "",
        "Full name": "Northwest Australia",
        "Mean Latitude": " (13°S)",
        "Rossby Radius": 100,
    },
    "westatlantic": {
        "lat_north": 38.7,
        "lat_south": 32.7,
        "lon_east": -73,
        "lon_west": -76,
        "reduce_xd_num": 1,
        "LLC4320_file": "",
        "simSWOT_file": "",
        "NEMO_file": "",
        "Full name": "West Atlantic",
        "Mean Latitude": " (35°N)",
        "Rossby Radius": 35,
    },
    "newcaledonia": {
        "lat_north": -22,
        "lat_south": -26,
        "lon_east": 171,
        "lon_west": 166,
        "reduce_xd_num": 1,
        "LLC4320_file": "",
        "simSWOT_file": "",
        "NEMO_file": "",
        "Full name": "New Caledonia",
        "Mean Latitude": " (24°S)",
        "Rossby Radius": "NA",
    },
    "labradorsea": {
        "lat_north": 63.79824,
        "lat_south": 59.5549,
        "lon_east": -58.52856,
        "lon_west": -63.61784,
        "reduce_xd_num": 1,
        "LLC4320_file": "",
        "simSWOT_file": "",
        "NEMO_file": "",
        "Full name": "Labrador Sea",
        "Mean Latitude": " (62°N)",
        "Rossby Radius": "NA",
    },
    # "Florida": {"lat_north": 29, "lat_south": 24, "lon_east": -77, "lon_west": -82, "reduce_xd_num": 1, "LLC4320_file": "", "SimSWOT_file": "", "NEMO_file": "", "Full name": "Florida", "Mean Latitude": " (26.5°N)", "Rossby Radius": 45},
    # "GrandBanks": {"lat_north": 40, "lat_south": 35, "lon_east": -45, "lon_west": -50, "reduce_xd_num": 1, "LLC4320_file": "", "SimSWOT_file": "", "NEMO_file": "", "Full name": "Grand Banks", "Mean Latitude": " (37.5°N)", "Rossby Radius": 30},
    # "Arbic": {"lat_north": 43, "lat_south": 27.5, "lon_east": -40, "lon_west": -60, "reduce_xd_num": 1, "LLC4320_file": "", "SimSWOT_file": "", "NEMO_file": "", "Full name": "Arbic", "Mean Latitude": " (35°N)", "Rossby Radius": 30},
    # "Equator": {"lat_north": 5, "lat_south": -5, "lon_east": -10, "lon_west": -35, "reduce_xd_num": 1, "LLC4320_file": "", "SimSWOT_file": "", "NEMO_file": "", "Full name": "Atlantic Equator", "Mean Latitude": " (0°)", "Rossby Radius": 200},
    # "Interior": {"lat_north": 32, "lat_south": 22, "lon_east": -30, "lon_west": -42, "reduce_xd_num": 1, "LLC4320_file": "", "SimSWOT_file": "", "NEMO_file": "", "Full name": "Atlantic Interior", "Mean Latitude": " (27°N)", "Rossby Radius": 40},
    # "GulfStream": {"southwest": (-80, 25), "southeast": (-35, 25), "northwest": (-80, 39), "northeast": (-35, 49), "reduce_xd_num": 1, "LLC4320_file": "", "SimSWOT_file": "", "NEMO_file": "", "Full name": "Gulf Stream", "Mean Latitude": " (37°N)", "Rossby Radius": 35},
}

# only use one region
# region_dict = {
#     "acc": {
#         "lat_north": -53,
#         "lat_south": -57.5,
#         "lon_east": 158,
#         "lon_west": 148,
#         "reduce_xd_num": 2,
#         "LLC4320_file": "",
#         "simSWOT_file": "",
#         "NEMO_file": "",
#         "Full name": "Antarctic Circumpolar Current",
#         "Mean Latitude": " (55°S)",
#         "Rossby Radius": 15,
#     },
# }

# Update the region_dict dynamically
for region, params in region_dict.items():
    try:
        # Merge files if data/MITgcm/{region}/*merged_dates_{mindate}-{maxdate}.nc" does not exist
        region_output_dir = f"../data/MITgcm/2026-06-01/{region}"
        mindate = "20110913"
        maxdate = "20121114"
        suffix = "timemean_removed_snapshot_mean_LLL_Bessels_tapered_CG" 

        if maxdate is None:
            # search for max date in the directory and set maxdate to max-1
            filenames = [f for f in os.listdir(region_output_dir) if f.startswith("2026") and f.endswith(".nc")]
            date_numbers = []
            for filename in filenames:
                match = re.search(rf"LLC4320_{region}_(\d+)_{suffix}\.nc", filename)
                if match:
                    date_numbers.append(int(match.group(1)))
            if date_numbers:
                max_date_found = max(date_numbers)
                print(f"Set max_date to {max_date_found} based on files in {region_output_dir}")
                maxdate = str(max_date_found)
            else:
                raise FileNotFoundError(f"No files found in {region_output_dir} to determine maxdate for region {region}.")


        merged_file = os.path.join(
            region_output_dir,
            f"LLC4320_{region}_merged_dates_{mindate}-{maxdate}_{suffix}.nc",
        )
        run_new = False 
        if not os.path.exists(merged_file) or run_new:
            if os.path.exists(merged_file):
                os.remove(merged_file)
                print(f"Deleted existing file: {merged_file}")
                
            merge_files_and_savenetcdf(
                region_output_dir,
                region,
                date=True,
                mindate=mindate,
                maxdate=maxdate,
                startswith="2026",
                suffix=suffix,
            )
        else:
            print(f"Merged file already exists for region {region}: {merged_file}")

        # Get the most recent LLC4320 file
        params["LLC4320_file"] = get_most_recent_file(
            region_name=region,
            file_pattern=f"../data/MITgcm/2026-06-01/{region}/*_{mindate}-{maxdate}_{suffix}.nc",
        )
    except FileNotFoundError as e:
        print(f"Warning: {e}. Skipping LLC4320 file for region: {region}")

    try:

        min_cycle = "001"
        max_cycle = "039"

        make_new = False  # Set to True to force new file creation, False to skip if file exists

        # Merge files if data/SWOT_L3/{region}/*_allcycles_{region}.nc" does not exist
        region_output_dir = f"../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/science_phase/{region}"
        suffix = "timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels"
        all_cycles_file = os.path.join(
            region_output_dir,
            f"SWOT_L3_{region}_cycles_{min_cycle}-{max_cycle}_{suffix}.nc",
        )
        print(all_cycles_file)
        if not os.path.exists(all_cycles_file) or make_new:
            merge_files_and_savenetcdf(
                region_output_dir,
                region,
                min_cycle=min_cycle,
                drop_cycles=None,
                max_cycle=max_cycle,
                startswith="2026",
                suffix=suffix,
            )
        else:
            print(
                f"All cycles file already exists for region {region}: {all_cycles_file}"
            )

        # Get most recently merged SWOT file
        params["SWOT_file"] = get_most_recent_file(
            region_name=region,
            file_pattern=f"../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/science_phase/{region}/*_{min_cycle}-{max_cycle}_{suffix}.nc",
        )
    except FileNotFoundError as e:
        print(f"Warning: {e}. Skipping SWOT file for region: {region}")
    except NotImplementedError as e:
        print(f"Warning: {e}. Skipping SWOT file for region: {region}")


    try:
        min_cycle = "478"
        max_cycle = None

        make_new = False  # Set to True to force new file creation, False to skip if file exists

        # Merge files if data/SWOT_L3/{region}/*_allcycles_{region}.nc" does not exist
        region_output_dir = f"../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/fast_phase/{region}"
        suffix = "timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels"

        if max_cycle is None:
            # search for max cycle in the directory and set max_cycle to max-1
            filenames = [f for f in os.listdir(region_output_dir) if f.startswith("2026") and f.endswith(".nc")]
            cycle_numbers = []
            for filename in filenames:
                match = re.search(rf"SWOT_L3_{region}_cycle_(\d+)_{suffix}\.nc", filename)
                if match:
                    cycle_numbers.append(int(match.group(1)))
            if cycle_numbers:
                max_cycle_found = max(cycle_numbers)
                print(f"Set max_cycle to {max_cycle_found} based on files in {region_output_dir}")
                max_cycle = str(max_cycle_found)
            else:
                raise FileNotFoundError(f"No files found in {region_output_dir} to determine max_cycle for region {region}.")

        all_cycles_file = os.path.join(
            region_output_dir,
            f"SWOT_L3_{region}_cycles_{min_cycle}-{max_cycle}_{suffix}.nc",
        )
        print(all_cycles_file)
        if not os.path.exists(all_cycles_file) or make_new:
            merge_files_and_savenetcdf(
                region_output_dir,
                region,
                min_cycle=min_cycle,
                drop_cycles=None,
                max_cycle=max_cycle,
                startswith="2026",
                suffix=suffix,
            )
        else:
            print(
                f"All cycles file already exists for region {region}: {all_cycles_file}"
            )

        # Get most recently merged SWOT file
        params["SWOT_fast_phase_file"] = get_most_recent_file(
            region_name=region,
            file_pattern=f"../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/fast_phase/{region}/*_{min_cycle}-{max_cycle}_{suffix}.nc",
        )
    except FileNotFoundError as e:
        print(f"Warning: {e}. Skipping SWOT file for region: {region}")


    try:
        cycle_list = None
        min_cycle = "001"
        max_cycle = "016"
        # Merge files if data/simSWOT/{region}/*_allcycles_{region}.nc" does not exist
        region_output_dir = f"../data/simSWOT/2026-05-27/science_phase/{region}"

        # if region == "acc":
        #     suffix = "timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG"
        # else:
        #     suffix = "timemean_removed_linear_fit_Bessels_tapered_CG"
        suffix = "timemean_removed_cycle_mean_LLL_Bessels_tapered_CG"

        if cycle_list:
            all_cycles_file = os.path.join(
                region_output_dir,
                f"simSWOT_{region}_cycles_{'-'.join(cycle_list)}_{suffix}.nc",
            )
            file_pattern = (
                f"../data/simSWOT/2026-05-27/science_phase/{region}/*_{'-'.join(cycle_list)}_{suffix}.nc"
            )
        elif min_cycle and max_cycle:
            all_cycles_file = os.path.join(
                region_output_dir,
                f"simSWOT_{region}_cycles_{min_cycle}-{max_cycle}_{suffix}.nc",
            )
            file_pattern = (
                f"../data/simSWOT/2026-05-27/science_phase/{region}/*_{min_cycle}-{max_cycle}_{suffix}.nc"
            )
        else:
            raise ValueError(
                "Either cycle_list or both min_cycle and max_cycle must be provided."
            )

        run_new = False  # Set to True to force new file creation, False to skip if file exists
        if run_new and os.path.exists(all_cycles_file):
            os.remove(all_cycles_file)
            print(f"Deleted existing file: {all_cycles_file}")

            # If the file was deleted, we should also set the file_pattern to look for the individual cycle files that will be merged to create the all_cycles_file
            if cycle_list:
                file_pattern = f"../data/simSWOT/2026-05-27/science_phase/{region}/*_{'-'.join(cycle_list)}_{suffix}.nc"
            elif min_cycle and max_cycle:
                file_pattern = f"../data/simSWOT/2026-05-27/science_phase/{region}/*_{min_cycle}-{max_cycle}_{suffix}.nc"
            else:
                raise ValueError(
                    "Either cycle_list or both min_cycle and max_cycle must be provided."
                )

        if not os.path.exists(all_cycles_file):

            if cycle_list:
                merge_files_and_savenetcdf(
                    region_output_dir,
                    region,
                    cycle_list=cycle_list,
                    startswith="2026",
                    SWOT_name="simSWOT",
                    suffix=suffix,
                )
            elif min_cycle and max_cycle:
                merge_files_and_savenetcdf(
                    region_output_dir,
                    region,
                    min_cycle=min_cycle,
                    max_cycle=max_cycle,
                    startswith="2026",
                    SWOT_name="simSWOT",
                    suffix=suffix,
                )
            else:
                raise ValueError(
                    "Either cycle_list or both min_cycle and max_cycle must be provided."
                )
        else:
            print(
                f"All cycles file already exists for region {region}: {all_cycles_file}"
            )

        # Get most recently merged SWOT file
        params["simSWOT_file"] = get_most_recent_file(
            region_name=region, file_pattern=file_pattern
        )
    except FileNotFoundError as e:
        print(f"Warning: {e}. Skipping simSWOT file for region: {region}")

# merge fast_phase files for  simSWOT
for region, params in region_dict.items():

    try:

        # Merge simSWOT files for fast_phase if not already merged
        region_output_dir = f"../data/simSWOT/2026-05-27/fast_phase/{region}"
        min_cycle = "001"
        max_cycle = None
        suffix = "timemean_removed_cycle_mean_LLL_Bessels_tapered_CG"

        if max_cycle is None:
            # search for max cycle in the directory and set max_cycle to max-1
            filenames = [f for f in os.listdir(region_output_dir) if f.startswith("2026") and f.endswith(".nc")]
            cycle_numbers = []
            for filename in filenames:
                match = re.search(rf"simSWOT_{region}_cycle_(\d+)_{suffix}\.nc", filename)
                if match:
                    cycle_numbers.append(int(match.group(1)))
            if cycle_numbers:
                max_cycle_found = f"{max(cycle_numbers):03d}"
                print(f"Set max_cycle to {int(max_cycle_found) - 1:03d} based on files in {region_output_dir}")
                max_cycle = f"{int(max_cycle_found) - 1:03d}"
            else:
                raise FileNotFoundError(f"No cycle files found in {region_output_dir} to determine max_cycle for region {region}.")

        all_cycles_file = os.path.join(
            region_output_dir,
            f"simSWOT_{region}_cycles_{min_cycle}-{max_cycle}_{suffix}.nc",
        )
        run_new = False  # Set to True to force new file creation, False to skip if file exists
        if run_new or not os.path.exists(all_cycles_file):
            if os.path.exists(all_cycles_file):
                os.remove(all_cycles_file)
                print(f"Deleted existing file: {all_cycles_file}")
            merge_files_and_savenetcdf(
                region_output_dir,
                region,
                min_cycle=min_cycle,
                max_cycle=max_cycle,
                startswith="2026",
                SWOT_name="simSWOT",
                suffix=suffix,
            )
        else:
            print(
                f"All cycles file already exists for region {region}: {all_cycles_file}"
            )          # Get most recently merged SWOT file 
        params["simSWOT_fast_phase_file"] = get_most_recent_file(
            region_name=region,
            file_pattern=f"../data/simSWOT/2026-05-27/fast_phase/{region}/*_{min_cycle}-{max_cycle}_{suffix}.nc",
        )
    except FileNotFoundError as e:
        print(f"Warning: {e}. Skipping simSWOT fast_phase file for region: {region}")

Merged file already exists for region acc: ../data/MITgcm/2026-06-01/acc/LLC4320_acc_merged_dates_20110913-20121114_timemean_removed_snapshot_mean_LLL_Bessels_tapered_CG.nc
../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/science_phase/acc/SWOT_L3_acc_cycles_001-039_timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels.nc
All cycles file already exists for region acc: ../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/science_phase/acc/SWOT_L3_acc_cycles_001-039_timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels.nc
Set max_cycle to 577 based on files in ../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/fast_phase/acc
../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/fast_phase/acc/SWOT_L3_acc_cycles_478-577_timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels.nc
All cycles file already exists for region acc: ../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/fast_phase/acc/SWOT_L3_acc_cycles_478-577_timemean_removed_cycle_mean_f

## MITgcm/SimSWOT/SWOT method plots

### Functions

In [ ]:
# 5x3 panel plot to plot 5 regions with 3 datasets each, and a toggle between which SF/CG/etc to show
def plot_5x3_panel(region_dict, plotting_variable='KE flux from ASF', confint=True, color='tab:blue', linestyle='-'):
    fig, axes = plt.subplots(5, 3, figsize=(15, 20), sharex=True)

    for i, (region, params) in enumerate(region_dict.items()):
        if i >= 5:
            break  # Only plot first 5 regions

        ds_LLC4320 = load_dataset("LLC4320_file", params, region, lambda ds: preprocess_LLC4320(ds, region))
        ds_simSWOT = load_dataset("SimSWOT_file", params, region)
        ds_SWOT = load_dataset("SWOT_file", params, region).mean(["num_lines","num_pixels"])

        # If no datasets are available, skip to the next region
        if all(ds is None for ds in [ds_LLC4320, ds_simSWOT, ds_SWOT]):
            continue

        if plotting_variable == 'KE flux from ASF':

            ds_LLC4320_var = -0.5 * ds_LLC4320['asf_mean'] * (1318 / 2000) ** 2
            ds_simSWOT_var = -0.5 * ds_simSWOT['asf_down']
            ds_SWOT_var = -0.5 * ds_SWOT['asf_down']

            ds_LLC4320_xvar = 2 * np.pi / ds_LLC4320['i_diffs']
            ds_simSWOT_xvar = 2 * np.pi / ds_simSWOT['num_lines_diffs']
            ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']

        if plotting_variable == 'Enstrophy flux from ASF':

            ds_LLC4320_var = 2 * ds_LLC4320['asf_mean'] * (1318 / 2000) ** 2 / ds_LLC4320['i_diffs']**2
            ds_simSWOT_var = 2 * ds_simSWOT['asf_down'] / ds_simSWOT['num_lines_diffs']**2
            ds_SWOT_var = 2 * ds_SWOT['asf_down'] / ds_SWOT['num_lines_diffs']**2

            ds_LLC4320_xvar = 2 * np.pi / ds_LLC4320['i_diffs']
            ds_simSWOT_xvar = 2 * np.pi / ds_simSWOT['num_lines_diffs']
            ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']

        if plotting_variable == 'Enstrophy flux from dwAw':

            ds_LLC4320_var = -0.5 * ds_LLC4320['asfq_mean'] * (1318 / 2000) ** 2
            ds_simSWOT_var = -0.5 * ds_simSWOT['asfq_down']
            ds_SWOT_var = -0.5 * ds_SWOT['asfq_down']

            ds_LLC4320_xvar = 2 * np.pi / ds_LLC4320['i_diffs']
            ds_simSWOT_xvar = 2 * np.pi / ds_simSWOT['num_lines_diffs']
            ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']

        if plotting_variable == 'KE flux from CG':

            ds_LLC4320_var = ds_LLC4320['EFlux_CG'] * (1318 / 2000) ** 2
            ds_simSWOT_var = ds_simSWOT['EFlux_CG']
            ds_SWOT_var = ds_SWOT['EFlux_CG']

            ds_LLC4320_xvar = ds_LLC4320['K_coarse_grain'].mean('time')
            ds_simSWOT_xvar = ds_simSWOT['K_coarse_grain'].mean('swath_num')
            ds_SWOT_xvar = ds_SWOT['K_coarse_grain'].mean('swath_num')
            

        if plotting_variable == 'Enstrophy flux from CG':

            ds_LLC4320_var = ds_LLC4320['QFlux_CG'] * (1318 / 2000) ** 2
            ds_simSWOT_var = ds_simSWOT['QFlux_CG']
            ds_SWOT_var = ds_SWOT['QFlux_CG']

            ds_LLC4320_xvar = ds_LLC4320['K_coarse_grain'].mean('time')
            ds_simSWOT_xvar = ds_simSWOT['K_coarse_grain'].mean('swath_num')
            ds_SWOT_xvar = ds_SWOT['K_coarse_grain'].mean('swath_num')

            
        axes[i, 0].semilogx(
            ds_LLC4320_xvar,
            ds_LLC4320_var.mean('time'),
            color=color,            
            linestyle=linestyle,
            label='LLC4320',
        )
        axes[i, 1].semilogx(
            ds_simSWOT_xvar,
            ds_simSWOT_var.mean('swath_num'),
            color=color,
            linestyle=linestyle,
            label='SimSWOT',
        )
        axes[i, 2].semilogx(
            ds_SWOT_xvar,
            ds_SWOT_var.mean('swath_num'),
            color=color,
            linestyle=linestyle,
            label='SWOT',
        )

        upper_quantile = 0.95
        lower_quantile = 0.05
        if confint:
            axes[i, 0].fill_between(
                ds_LLC4320_xvar,
                ds_LLC4320_var.quantile(lower_quantile, dim='time'),
                ds_LLC4320_var.quantile(upper_quantile, dim='time'),
                color=color, alpha=0.2,
            )
            axes[i, 1].fill_between(
                ds_simSWOT_xvar,
                ds_simSWOT_var.quantile(lower_quantile, dim='swath_num'),
                ds_simSWOT_var.quantile(upper_quantile, dim='swath_num'),
                color=color, alpha=0.2
            )
            axes[i, 2].fill_between(
                ds_SWOT_xvar,
                ds_SWOT_var.quantile(lower_quantile, dim='swath_num'),
                ds_SWOT_var.quantile(upper_quantile, dim='swath_num'),
                color=color, alpha=0.2
            )

        # Add text with full region name in the top left corner of the first column
        axes[i, 0].text(0.05, 0.95, params["Full name"], transform=axes[i, 0].transAxes,
                        fontsize=10, verticalalignment='top', fontweight='bold',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
        
        # Add text with dataset name in the bottom left corner of each panel
        axes[i, 0].text(0.05, 0.05, 'LLC4320', transform=axes[i, 0].transAxes,
                        fontsize=10, verticalalignment='bottom',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
        axes[i, 1].text(0.05, 0.05, 'SimSWOT', transform=axes[i, 1].transAxes,
                        fontsize=10, verticalalignment='bottom',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
        axes[i, 2].text(0.05, 0.05, 'SWOT', transform=axes[i, 2].transAxes,
                        fontsize=10, verticalalignment='bottom',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))

        # Set ylim/ymax to be the same for all panels in a row based on the ylim/ymax of LLC4320 only



        for j in range(3):
            axes[i, j].set_ylim(axes[i, 0].get_ylim())
            axes[i, j].set_ylabel(plotting_variable)
            axes[i, j].set_xlabel("Wavenumber [m$^{-1}$]")
            axes[i, j].set_xlim(1e-6, 1e-2)  # Set x-limits for all axes
            axes[i, j].hlines(0, 1e-6, 1e-2, colors="k", lw=1, zorder=0)
            secax = axes[i, j].secondary_xaxis('top', functions=(wavenumber_to_distance, distance_to_wavenumber))
            secax.set_xlabel("Separation Distance [m]")
            secax.set_xlim(wavenumber_to_distance(1e-2), wavenumber_to_distance(1e-6))  # Match limits to primary x-axis
            secax.tick_params(direction="in", which="both", bottom=False)
            
    plt.tight_layout()

### Plotting

In [ ]:
# Run the 5x3 panel plot function

sns.set_context("talk")

var_dict = {
    'KE flux from ASF': ('tab:blue', '-'),
    'Enstrophy flux from ASF': ('tab:red', '-'),
    'Enstrophy flux from dwAw': ('tab:red', '-'),
    'KE flux from CG': ('tab:blue', '-'),
    'Enstrophy flux from CG': ('tab:red', '-'),
}

for plotting_variable, (color, linestyle) in var_dict.items():
    print(f"Plotting {plotting_variable}")
    plot_5x3_panel(region_dict, plotting_variable=plotting_variable, color=color, linestyle=linestyle)
    plt.savefig(f"figs/{plotting_variable.replace(' ', '_')}_MITgcm-SimSWOT-SWOT_5x3_panel_talk.png", dpi=300)

## Prelim solo region plots

### Functions

In [ ]:
# 1x3 panel plot to plot 1 region with 3 datasets each, and a toggle between which SF/CG/etc to show
def plot_1x3_panel(region_dict, plotting_variable='KE flux from ASF', confint=True, color='tab:blue', linestyle='-', KE_ymin=None, KE_ymax=None, Q_ymin=None, Q_ymax=None, convert_Wm3=False):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True)

    region, params = list(region_dict.items())[0]

    ds_LLC4320 = load_dataset("LLC4320_file", params, region, lambda ds: preprocess_LLC4320(ds, region))
    ds_simSWOT = load_dataset("SimSWOT_file", params, region)
    ds_SWOT = load_dataset("SWOT_file", params, region)

    if plotting_variable == 'KE flux from ASF':

        ds_LLC4320_var = -0.5 * ds_LLC4320['asf_mean'] * (1318 / 2000) ** 2
        ds_simSWOT_var = -0.5 * ds_simSWOT['asf_down']
        ds_SWOT_var = -0.5 * ds_SWOT['asf_down']

        ds_LLC4320_xvar = 2 * np.pi / ds_LLC4320['i_diffs']
        ds_simSWOT_xvar = 2 * np.pi / ds_simSWOT['num_lines_diffs']
        ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']

    if plotting_variable == 'Enstrophy flux from ASF':

        ds_LLC4320_var = 2 * ds_LLC4320['asf_mean'] * (1318 / 2000) ** 2 / ds_LLC4320['i_diffs']**2
        ds_simSWOT_var = 2 * ds_simSWOT['asf_down'] / ds_simSWOT['num_lines_diffs']**2
        ds_SWOT_var = 2 * ds_SWOT['asf_down'] / ds_SWOT['num_lines_diffs']**2

        ds_LLC4320_xvar = 2 * np.pi / ds_LLC4320['i_diffs']
        ds_simSWOT_xvar = 2 * np.pi / ds_simSWOT['num_lines_diffs']
        ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']

    if plotting_variable == 'Enstrophy flux from dwAw':

        ds_LLC4320_var = -0.5 * ds_LLC4320['asfq_mean'] * (1318 / 2000) ** 2
        ds_simSWOT_var = -0.5 * ds_simSWOT['asfq_down']
        ds_SWOT_var = -0.5 * ds_SWOT['asfq_down']

        ds_LLC4320_xvar = 2 * np.pi / ds_LLC4320['i_diffs']
        ds_simSWOT_xvar = 2 * np.pi / ds_simSWOT['num_lines_diffs']
        ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']

    if plotting_variable == 'KE flux from CG':

        ds_LLC4320_var = ds_LLC4320['EFlux_CG'] * (1318 / 2000) ** 2
        ds_simSWOT_var = ds_simSWOT['EFlux_CG']
        ds_SWOT_var = ds_SWOT['EFlux_CG']

        ds_LLC4320_xvar = ds_LLC4320['K_coarse_grain'].mean('time')
        ds_simSWOT_xvar = ds_simSWOT['K_coarse_grain'].mean('swath_num')
        ds_SWOT_xvar = ds_SWOT['K_coarse_grain'].mean('swath_num')
        

    if plotting_variable == 'Enstrophy flux from CG':

        ds_LLC4320_var = ds_LLC4320['QFlux_CG'] * (1318 / 2000) ** 2
        ds_simSWOT_var = ds_simSWOT['QFlux_CG']
        ds_SWOT_var = ds_SWOT['QFlux_CG']

        ds_LLC4320_xvar = ds_LLC4320['K_coarse_grain'].mean('time')
        ds_simSWOT_xvar = ds_simSWOT['K_coarse_grain'].mean('swath_num')
        ds_SWOT_xvar = ds_SWOT['K_coarse_grain'].mean('swath_num')

    if convert_Wm3:

        ds_LLC4320_var *= 1025
        ds_simSWOT_var *= 1025
        ds_SWOT_var *= 1025

    axes[0].semilogx(
        ds_LLC4320_xvar,
        ds_LLC4320_var.mean('time'),
        color=color,            
        linestyle=linestyle,
        label='LLC4320',
    )
    axes[1].semilogx(
        ds_simSWOT_xvar,
        ds_simSWOT_var.mean('swath_num'),
        color=color,
        linestyle=linestyle,
        label='SimSWOT',
    )
    axes[2].semilogx(
        ds_SWOT_xvar,
        ds_SWOT_var.mean('swath_num'),
        color=color,
        linestyle=linestyle,
        label='SWOT',
    )

    upper_quantile = 0.95
    lower_quantile = 0.05
    if confint:
        axes[0].fill_between(
            ds_LLC4320_xvar,
            ds_LLC4320_var.quantile(lower_quantile, dim='time'),
            ds_LLC4320_var.quantile(upper_quantile, dim='time'),
            color=color, alpha=0.2,
        )
        axes[1].fill_between(
            ds_simSWOT_xvar,
            ds_simSWOT_var.quantile(lower_quantile, dim='swath_num'),
            ds_simSWOT_var.quantile(upper_quantile, dim='swath_num'),
            color=color, alpha=0.2
        )
        axes[2].fill_between(
            ds_SWOT_xvar,
            ds_SWOT_var.quantile(lower_quantile, dim='swath_num'),
            ds_SWOT_var.quantile(upper_quantile, dim='swath_num'),
            color=color, alpha=0.2
        )

    # Add text with full region name in the top left corner of the first column
    axes[0].text(0.05, 0.95, params["Full name"], transform=axes[0].transAxes,
                    fontsize=10, verticalalignment='top', fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    
    # Add text with dataset name in the bottom left corner of each panel
    axes[0].text(0.05, 0.05, 'LLC4320', transform=axes[0].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    axes[1].text(0.05, 0.05, 'SimSWOT', transform=axes[1].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    axes[2].text(0.05, 0.05, 'SWOT', transform=axes[2].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))

    # Set ylim/ymax to be the same for all panels in a row based on the ylim/ymax of LLC4320 only
    for i in range(3):
        if any(v is not None for v in [KE_ymin, KE_ymax]) and 'KE flux' in plotting_variable:
            axes[i].set_ylim(KE_ymin, KE_ymax)
        elif any(v is not None for v in [Q_ymin, Q_ymax]) and 'Enstrophy flux' in plotting_variable:
            axes[i].set_ylim(Q_ymin, Q_ymax)
        else:
            axes[i].set_ylim(axes[0].get_ylim())

        if 'KE flux' in plotting_variable and i == 0:
            if convert_Wm3:
                plotting_variable = plotting_variable + r" [W m$^{-3}$]"
            else:
                plotting_variable = plotting_variable + r" [m$^2$ s$^{-3}$]"

        elif 'Enstrophy flux' in plotting_variable and i == 0:
            plotting_variable = plotting_variable + r" [s$^{-3}$]"
        axes[0].set_ylabel(plotting_variable)
        axes[i].set_xlabel("Wavenumber [m$^{-1}$]")
        axes[i].set_xlim(1e-6, 1e-2)  # Set x-limits for all axes
        axes[i].hlines(0, 1e-6, 1e-2, colors="k", lw=1, zorder=0)
        secax = axes[i].secondary_xaxis('top', functions=(wavenumber_to_distance, distance_to_wavenumber))
        secax.set_xlabel("Separation Distance [m]")
        secax.set_xlim(wavenumber_to_distance(1e-2), wavenumber_to_distance(1e-6))  # Match limits to primary x-axis
        secax.tick_params(direction="in", which="both", bottom=False)
        
    plt.tight_layout()

### Plotting

In [ ]:
# Run the 1x3 panel plot function

sns.set_context("notebook")

var_dict = {
    'KE flux from ASF': ('tab:blue', '-'),
    'Enstrophy flux from ASF': ('tab:red', '-'),
    'Enstrophy flux from dwAw': ('tab:red', '-'),
    'KE flux from CG': ('tab:blue', '-'),
    'Enstrophy flux from CG': ('tab:red', '-'),
}

for plotting_variable, (color, linestyle) in var_dict.items():
    print(f"Plotting {plotting_variable}")
    plot_1x3_panel(region_dict, plotting_variable=plotting_variable, color=color, linestyle=linestyle, KE_ymin=-6e-6, KE_ymax=1e-6, Q_ymin=-4e-15, Q_ymax=2e-15, convert_Wm3=True)
    # plt.savefig(f"figs/{plotting_variable.replace(' ', '_')}_MITgcm-SimSWOT-SWOT_1x3_panel_ACC.png", dpi=300)

## 2026-04-06 paper 1 single region multi method multi dataset panels

### Functions

In [ ]:
def plot_1x3_allmethods(region_dict, region_idx=0, confint=True, ASF=True, CG=True, Bessels=False, LLL=False, KE=True, KE_ymin=None, KE_ymax=None, Bessel_mean_bug=True, filter_outliers=False,use_km=False, convert_KEFlux_to_wattperm3=False, season=None, phase="both", asf_dir="mean", LLL_dir="mean", scale=1, zoom=None):

    region, params = list(region_dict.items())[region_idx]

    ds_LLC4320 = load_dataset("LLC4320_file", params, region)
    ds_LLC4320 = ds_LLC4320.assign_coords(time=pd.date_range(start="2011-09-13", periods=ds_LLC4320.sizes['time'], freq='h'))
    simSWOT_ds_list = []
    SWOT_ds_list = []

    if phase == "both" or phase == "fast":
        ds_simSWOT_fast = load_dataset("simSWOT_fast_phase_file", params, region)
        ds_SWOT_fast = load_dataset("SWOT_fast_phase_file", params, region)

        simSWOT_ds_list.append(ds_simSWOT_fast)
        SWOT_ds_list.append(ds_SWOT_fast)

    if phase == "both" or phase == "science":
        ds_simSWOT_science = load_dataset("simSWOT_file", params, region)
        ds_SWOT_science = load_dataset("SWOT_file", params, region)

        simSWOT_ds_list.append(ds_simSWOT_science)
        SWOT_ds_list.append(ds_SWOT_science)


    for i, ds in enumerate(simSWOT_ds_list):
        ds = ds.assign_coords(time_coord=ds['time'].mean(["num_lines", "num_pixels"]))
        simSWOT_ds_list[i] = ds
    for i, ds in enumerate(SWOT_ds_list):
        try:
            ds = ds.assign_coords(time_coord=ds['time'].mean(["num_lines", "num_pixels"]))    
            SWOT_ds_list[i] = ds
        except Exception as e:
            if i == 0:
                print(f"Warning: Could not assign time_coord to fast phase SWOT dataset for region {region} due to error: {e}. This may be because the fast phase dataset is missing or has a different structure. Skipping time_coord assignment for this dataset.")
            if i == 1:
                print(f"Warning: Could not assign time_coord to science phase SWOT dataset for region {region} due to error: {e}. This may be because the science phase dataset is missing or has a different structure. Skipping time_coord assignment for this dataset.")
                SWOT_ds_list[i] = None


    if season is not None:
        # cut datasets to season months where season is a list of month integers, e.g. [12, 1, 2] for DJF
        ds_LLC4320 = ds_LLC4320.sel(time=ds_LLC4320['time'].dt.month.isin(season))

        for i in range(len(simSWOT_ds_list)):
            simSWOT_ds_list[i] = simSWOT_ds_list[i].sel(time_coord=simSWOT_ds_list[i]['time_coord'].dt.month.isin(season))
        for i in range(len(SWOT_ds_list)):     
            SWOT_ds_list[i] = SWOT_ds_list[i].sel(time_coord=SWOT_ds_list[i]['time_coord'].dt.month.isin(season))

        # if any of the datasets are empty after seasonal selection, raise an error
        if len(ds_LLC4320['time']) == 0 or any(len(ds['time_coord']) == 0 for ds in simSWOT_ds_list) or any(len(ds['time_coord']) == 0 for ds in SWOT_ds_list):
            raise ValueError(f"No data left after seasonal selection for region {region} and season {season}. Please check the datasets and season selection.")
        
    # Set flux variables
    if KE is True:

        ds_LLC4320['KE_asf'] = -0.5 * ds_LLC4320['asf_mean'] 
        ds_LLC4320['KE_CG'] = ds_LLC4320['EFlux_CG']
        ds_LLC4320['KE_LLL'] = -(2/(3 * ds_LLC4320['i_diffs'])) * ds_LLC4320['LLL_mean'] if LLL and 'LLL_mean' in ds_LLC4320.data_vars else None
        ds_LLC4320["KE_Bessel"] = (sum(ds_LLC4320[var] for var in ds_LLC4320.data_vars if "Bessel_asf" in var)/4) if Bessel_mean_bug else None

        for i in range(len(simSWOT_ds_list)):
            simSWOT_ds_list[i]['KE_asf'] = -0.5 * simSWOT_ds_list[i][f'asf_{asf_dir}']
            simSWOT_ds_list[i]['KE_CG'] = simSWOT_ds_list[i]['EFlux_CG'] 
            simSWOT_ds_list[i]['KE_LLL'] = -(2/(3 * simSWOT_ds_list[i]['num_lines_diffs'])) * simSWOT_ds_list[i][f'LLL_{LLL_dir}'] if LLL  and 'LLL_mean' in simSWOT_ds_list[i].data_vars else None
            simSWOT_ds_list[i]['KE_Bessel'] = (sum(simSWOT_ds_list[i][var] for var in simSWOT_ds_list[i].data_vars if "Bessel_asf" in var)/4) if Bessel_mean_bug else None

        for i in range(len(SWOT_ds_list)):        
            SWOT_ds_list[i]['KE_asf'] = -0.5 * SWOT_ds_list[i][f'asf_{asf_dir}'] 
            SWOT_ds_list[i]['KE_CG'] = SWOT_ds_list[i]['EFlux_CG']
            SWOT_ds_list[i]['KE_LLL'] = -(2/(3 * SWOT_ds_list[i]['num_lines_diffs'])) * SWOT_ds_list[i][f'LLL_{LLL_dir}'] if LLL  and 'LLL_mean' in SWOT_ds_list[i].data_vars else None
            SWOT_ds_list[i]['KE_Bessel'] = (sum(SWOT_ds_list[i][var] for var in SWOT_ds_list[i].data_vars if "Bessel_asf" in var)/4) if Bessel_mean_bug else None

        # ds_LLC4320_enst_asf = 2 * ds_LLC4320['asf_mean'] * (1318 / 2000) ** 2 / ds_LLC4320['i_diffs']**2
        # ds_simSWOT_enst_asf = 2 * ds_simSWOT['asf_down'] / ds_simSWOT['num_lines_diffs']**2
        # ds_SWOT_enst_asf = 2 * ds_SWOT['asf_down'] / ds_SWOT['num_lines_diffs']**2

        # print ratio of minimum KE flux from ASF to minimum KE flux from CG
        # print(f"Ratio of minimum KE flux from ASF to minimum KE flux from CG for LLC4320: {ds_LLC4320['KE_asf'].mean(dim='time').min().item() / ds_LLC4320['KE_CG'].mean(dim='time').min().item()}")
        # print(f"Ratio of minimum KE flux from ASF to minimum KE flux from CG for SimSWOT: {simSWOT_ds_list[0]['KE_asf'].mean(dim='swath_num').min().item() / simSWOT_ds_list[0]['KE_CG'].mean(dim='swath_num').min().item()}")
        # print(f"Ratio of minimum KE flux from ASF to minimum KE flux from CG for SWOT: {SWOT_ds_list[0]['KE_asf'].mean(dim='swath_num').min().item() / SWOT_ds_list[0]['KE_CG'].mean(dim='swath_num').min().item()}")

        if convert_KEFlux_to_wattperm3:
            scale_factor = 40  # Convert from m^2/s^3 to W/m^3 by multiplying by density of seawater
            ds_LLC4320['KE_asf'] = ds_LLC4320['KE_asf'] * scale_factor
            ds_LLC4320['KE_CG'] = ds_LLC4320['KE_CG'] * scale_factor
            ds_LLC4320['KE_LLL'] = ds_LLC4320['KE_LLL'] * scale_factor if LLL and 'KE_LLL' in ds_LLC4320.data_vars else None
            ds_LLC4320['KE_Bessel'] = ds_LLC4320['KE_Bessel'] * scale_factor if Bessel_mean_bug and 'KE_Bessel' in ds_LLC4320.data_vars else None
            for i in range(len(simSWOT_ds_list)):
                simSWOT_ds_list[i]['KE_asf'] = simSWOT_ds_list[i]['KE_asf'] * scale_factor if ASF else None
                simSWOT_ds_list[i]['KE_CG'] = simSWOT_ds_list[i]['KE_CG'] * scale_factor if CG else None
                simSWOT_ds_list[i]['KE_LLL'] = simSWOT_ds_list[i]['KE_LLL'] * scale_factor if LLL and 'LLL_mean' in simSWOT_ds_list[i].data_vars else None
                simSWOT_ds_list[i]['KE_Bessel'] = simSWOT_ds_list[i]['KE_Bessel'] * scale_factor if Bessel_mean_bug and 'KE_Bessel' in simSWOT_ds_list[i].data_vars else None

            for i in range(len(SWOT_ds_list)):            
                SWOT_ds_list[i]['KE_asf'] = SWOT_ds_list[i]['KE_asf'] * scale_factor if ASF else None
                SWOT_ds_list[i]['KE_CG'] = SWOT_ds_list[i]['KE_CG'] * scale_factor if CG else None
                SWOT_ds_list[i]['KE_LLL'] = SWOT_ds_list[i]['KE_LLL'] * scale_factor if LLL and 'LLL_mean' in SWOT_ds_list[i].data_vars else None
                SWOT_ds_list[i]['KE_Bessel'] = SWOT_ds_list[i]['KE_Bessel'] * scale_factor if Bessel_mean_bug and 'KE_Bessel' in SWOT_ds_list[i].data_vars else None

    # TODO LATER
    # if Q is True:

    #     ds_LLC4320_enst_dwAw = -0.5 * ds_LLC4320['asfq_mean'] * (1318 / 2000) ** 2
    #     ds_simSWOT_enst_dwAw = -0.5 * ds_simSWOT['asfq_down']
    #     ds_SWOT_enst_dwAw = -0.5 * ds_SWOT['asfq_down']

    #     ds_LLC4320_enst_CG = ds_LLC4320['QFlux_CG'] * (1318 / 2000) ** 2
    #     ds_simSWOT_enst_CG = ds_simSWOT['QFlux_CG']
    #     ds_SWOT_enst_CG = ds_SWOT['QFlux_CG']



    # Set x/K variables
    ds_LLC4320["Kvar_asf"] = 1 / ds_LLC4320['i_diffs']
    ds_LLC4320["Kvar_CG"] = ds_LLC4320['K_coarse_grain'].mean('time') / np.pi
    ds_LLC4320["Kvar_Bessel"] = ds_LLC4320['K'] if 'K' in ds_LLC4320.coords else None
    for i in range(len(simSWOT_ds_list)):
        simSWOT_ds_list[i]["Kvar_asf"] = 1 / simSWOT_ds_list[i]['num_lines_diffs']
        simSWOT_ds_list[i]["Kvar_CG"] = simSWOT_ds_list[i]['K_coarse_grain'].mean('swath_num') / np.pi
        simSWOT_ds_list[i]["Kvar_Bessel"] = simSWOT_ds_list[i]['K'] if 'K' in simSWOT_ds_list[i].coords else None

    for i in range(len(SWOT_ds_list)):            
        SWOT_ds_list[i]["Kvar_asf"] = 1 / SWOT_ds_list[i]['num_lines_diffs']
        SWOT_ds_list[i]["Kvar_CG"] = SWOT_ds_list[i]['K_coarse_grain'].mean('swath_num') / np.pi
        SWOT_ds_list[i]["Kvar_Bessel"] = SWOT_ds_list[i]['K'] if 'K' in SWOT_ds_list[i].coords else None

    # ds_LLC4320_Kvar_Bessel = ds_LLC4320['K']
    # ds_simSWOT_Kvar_Bessel = ds_simSWOT['K']
    # ds_SWOT_Kvar_Bessel = ds_SWOT['K']

    if use_km:
        ds_LLC4320["Kvar_asf"] *= 1e3
        ds_LLC4320["Kvar_CG"] *= 1e3
        ds_LLC4320["Kvar_Bessel"] = ds_LLC4320['Kvar_Bessel'] * 1e3 if 'K' in ds_LLC4320.coords else None
        for i in range(len(simSWOT_ds_list)):
            simSWOT_ds_list[i]["Kvar_asf"] *= 1e3
            simSWOT_ds_list[i]["Kvar_CG"] *= 1e3
            simSWOT_ds_list[i]["Kvar_Bessel"] = simSWOT_ds_list[i]['Kvar_Bessel'] * 1e3 if 'K' in simSWOT_ds_list[i].coords else None
        for i in range(len(SWOT_ds_list)):            
            SWOT_ds_list[i]["Kvar_asf"] *= 1e3
            SWOT_ds_list[i]["Kvar_CG"] *= 1e3
            SWOT_ds_list[i]["Kvar_Bessel"] = SWOT_ds_list[i]['Kvar_Bessel'] * 1e3 if 'K' in SWOT_ds_list[i].coords else None

    # if filter_outliers:
    # # Filter out outliers based on percentiles of the KE values for each dataset and set the newly filtered KE values to the original variable so that the plotting code doesn't need to be changed
    #     for ds, KE_var in zip([ds_LLC4320, *simSWOT_ds_list, *SWOT_ds_list],                            [ds_LLC4320['KE_asf'], *[ds['KE_asf'] for ds in simSWOT_ds_list], *[ds['KE_asf'] for ds in SWOT_ds_list]]):
    #         lower_bound = KE_var.quantile(0.1)
    #         upper_bound = KE_var.quantile(0.9)
    #         KE_var_filtered = KE_var.where((KE_var >= lower_bound) & (KE_var <= upper_bound))
    #         KE_var.data = KE_var_filtered.data

    # Plotting parameters
    asf_color = 'tab:red'
    asf_linewidth = 2
    asf_linestyle = 'solid'

    CG_color = 'k'
    CG_linewidth = 3
    CG_linestyle = 'solid'

    Bessel_color = 'tab:red'
    Bessel_linewidth = 2
    Bessel_linestyle = '--'

    LLL_color = 'tab:purple'
    LLL_linestyle = '-.'
    LLL_linewidth = 2

    fig, axes = plt.subplots(1, 1 + len(simSWOT_ds_list) + len(SWOT_ds_list), figsize=(5 * (1 + len(simSWOT_ds_list) + len(SWOT_ds_list)), 5), sharex=True, sharey=False)

    # print shape for bessel plot to check the dimensions are correct for plotting
    # if Bessels and 'KE_Bessel' in ds_LLC4320.data_vars:
    #     print(f"LLC4320 KE_Bessel shape: {ds_LLC4320['KE_Bessel'].shape}")
    #     print(f"LLC4320 Kvar_Bessel shape: {ds_LLC4320['Kvar_Bessel'].shape}")
    #     print(f"LLC4320 K dimsions: {ds_LLC4320['K'].dims if 'K' in ds_LLC4320.coords else 'K not in coords'}")
    #     print(f"LLC4320 variables: {list(ds_LLC4320.data_vars)}")

    # Plot LLC4320 variables
    axes[0].semilogx(ds_LLC4320["Kvar_CG"], ds_LLC4320["KE_CG"].mean('time'),c=CG_color,ls=CG_linestyle,lw=CG_linewidth) if CG else None
    axes[0].semilogx(ds_LLC4320["Kvar_asf"], ds_LLC4320["KE_asf"].mean('time')/scale,c=asf_color,ls=asf_linestyle, lw=asf_linewidth) if ASF else None
    axes[0].semilogx(ds_LLC4320["Kvar_Bessel"], ds_LLC4320["KE_Bessel"].mean('time')/scale,c=Bessel_color,ls=Bessel_linestyle, lw=Bessel_linewidth) if Bessels and 'KE_Bessel' in ds_LLC4320.data_vars else None
    axes[0].semilogx(ds_LLC4320["Kvar_asf"], ds_LLC4320["KE_LLL"].mean('time')/scale,c=LLL_color,ls=LLL_linestyle, lw=LLL_linewidth) if LLL and 'LLL_mean' in ds_LLC4320.data_vars else None

    # Plot simSWOT variables
    for i in range(len(simSWOT_ds_list)):
        axes_val_simSWOT = 1 if i == 0 else 2

        axes[axes_val_simSWOT].semilogx(simSWOT_ds_list[i]["Kvar_CG"], simSWOT_ds_list[i]["KE_CG"].mean('swath_num'),c=CG_color,ls=CG_linestyle,lw=CG_linewidth, label='CG') if CG else None
        axes[axes_val_simSWOT].semilogx(simSWOT_ds_list[i]["Kvar_asf"], simSWOT_ds_list[i]["KE_asf"].mean('swath_num')/scale,c=asf_color,ls=asf_linestyle, lw=asf_linewidth, label='ASF') if ASF else None
        axes[axes_val_simSWOT].semilogx(simSWOT_ds_list[i]["Kvar_asf"], simSWOT_ds_list[i]["KE_LLL"].mean('swath_num')/scale,c=LLL_color,ls=LLL_linestyle, lw=LLL_linewidth, label='LLL') if LLL and 'LLL_mean' in simSWOT_ds_list[i].data_vars else None
        axes[axes_val_simSWOT].semilogx(simSWOT_ds_list[i]["Kvar_Bessel"], simSWOT_ds_list[i]["KE_Bessel"].mean('swath_num')/scale,c=Bessel_color,ls=Bessel_linestyle, lw=Bessel_linewidth, label='Bessel') if Bessels and 'KE_Bessel' in simSWOT_ds_list[i].data_vars else None


    # Plot SWOT variables
    for i in range(len(SWOT_ds_list)):
        axes_val_swot = axes_val_simSWOT + 1 if i == 0 else axes_val_simSWOT + 2
        
        axes[axes_val_swot].semilogx(SWOT_ds_list[i]["Kvar_CG"], SWOT_ds_list[i]["KE_CG"].mean('swath_num'),c=CG_color,ls=CG_linestyle,lw=CG_linewidth, label='CG') if CG else None
        axes[axes_val_swot].semilogx(SWOT_ds_list[i]["Kvar_asf"], SWOT_ds_list[i]["KE_asf"].mean('swath_num')/scale,c=asf_color,ls=asf_linestyle, lw=asf_linewidth, label='ASF') if ASF else None
        axes[axes_val_swot].semilogx(SWOT_ds_list[i]["Kvar_asf"], SWOT_ds_list[i]["KE_LLL"].mean('swath_num')/scale,c=LLL_color,ls=LLL_linestyle, lw=LLL_linewidth, label='LLL') if LLL and 'LLL_mean' in SWOT_ds_list[i].data_vars else None
        axes[axes_val_swot].semilogx(SWOT_ds_list[i]["Kvar_Bessel"], SWOT_ds_list[i]["KE_Bessel"].mean('swath_num')/scale,c=Bessel_color,ls=Bessel_linestyle, lw=Bessel_linewidth, label='Bessel') if Bessels and 'KE_Bessel' in SWOT_ds_list[i].data_vars else None
    if confint:
        upper_quantile = 0.90
        lower_quantile = 0.1
        # corresponds to a 80% confidence interval

        # Plot LLC4320 confidence intervals
        axes[0].fill_between(
            ds_LLC4320["Kvar_asf"], 
            ds_LLC4320["KE_asf"].quantile(lower_quantile, dim='time'),
            ds_LLC4320["KE_asf"].quantile(upper_quantile, dim='time'),
            color=asf_color, alpha=0.2
                            ) if ASF else None
        axes[0].fill_between(
            ds_LLC4320["Kvar_CG"], 
            ds_LLC4320["KE_CG"].quantile(lower_quantile, dim='time'),
            ds_LLC4320["KE_CG"].quantile(upper_quantile, dim='time'),
            color=CG_color, alpha=0.2
                            ) if CG else None
        axes[0].fill_between(
            ds_LLC4320["Kvar_Bessel"], 
            ds_LLC4320["KE_Bessel"].quantile(lower_quantile, dim='time'),
            ds_LLC4320["KE_Bessel"].quantile(upper_quantile, dim='time'),
            color=Bessel_color, alpha=0.2
                            ) if Bessels else None      

        axes[0].fill_between(
            ds_LLC4320["Kvar_asf"], 
            ds_LLC4320["KE_LLL"].quantile(lower_quantile, dim='time'),
            ds_LLC4320["KE_LLL"].quantile(upper_quantile, dim='time'),
            color='tab:green', alpha=0.2
                            ) if LLL and 'LLL_mean' in ds_LLC4320.data_vars else None

        # Plot simSWOT confidence intervals
        for i in range(len(simSWOT_ds_list)):
            axes_val_simSWOT = 1 if i == 0 else 2
            axes[axes_val_simSWOT].fill_between(
                simSWOT_ds_list[i]["Kvar_asf"],
                simSWOT_ds_list[i]["KE_asf"].quantile(lower_quantile, dim='swath_num'),
                simSWOT_ds_list[i]["KE_asf"].quantile(upper_quantile, dim='swath_num'),
                color=asf_color, alpha=0.2
            ) if ASF else None
            axes[axes_val_simSWOT].fill_between(
                simSWOT_ds_list[i]["Kvar_CG"],
                simSWOT_ds_list[i]["KE_CG"].quantile(lower_quantile, dim='swath_num'),
                simSWOT_ds_list[i]["KE_CG"].quantile(upper_quantile, dim='swath_num'),
                color=CG_color, alpha=0.2
            ) if CG else None
            axes[axes_val_simSWOT].fill_between(
                simSWOT_ds_list[i]["Kvar_asf"],
                simSWOT_ds_list[i]["KE_LLL"].quantile(lower_quantile, dim='swath_num'),
                simSWOT_ds_list[i]["KE_LLL"].quantile(upper_quantile, dim='swath_num'),
                color=LLL_color, alpha=0.2
            ) if LLL and 'LLL_mean' in simSWOT_ds_list[i].data_vars else None
            axes[axes_val_simSWOT].fill_between(
                simSWOT_ds_list[i]["Kvar_Bessel"],
                simSWOT_ds_list[i]["KE_Bessel"].quantile(lower_quantile, dim='swath_num'),
                simSWOT_ds_list[i]["KE_Bessel"].quantile(upper_quantile, dim='swath_num'),
                color=Bessel_color, alpha=0.2
            ) if Bessels and 'KE_Bessel' in simSWOT_ds_list[i].data_vars else None

        # Plot SWOT confidence intervals
        for i in range(len(SWOT_ds_list)):            
            axes_val_swot = axes_val_simSWOT + 1 if i == 0 else axes_val_simSWOT + 2
            axes[axes_val_swot].fill_between(
                SWOT_ds_list[i]["Kvar_asf"],
                SWOT_ds_list[i]["KE_asf"].quantile(lower_quantile, dim='swath_num'),
                SWOT_ds_list[i]["KE_asf"].quantile(upper_quantile, dim='swath_num'),
                color=asf_color, alpha=0.2
            ) if ASF else None
            axes[axes_val_swot].fill_between(
                SWOT_ds_list[i]["Kvar_CG"],
                SWOT_ds_list[i]["KE_CG"].quantile(lower_quantile, dim='swath_num'),
                SWOT_ds_list[i]["KE_CG"].quantile(upper_quantile, dim='swath_num'),
                color=CG_color, alpha=0.2
            ) if CG else None
            axes[axes_val_swot].fill_between(
                SWOT_ds_list[i]["Kvar_asf"],
                SWOT_ds_list[i]["KE_LLL"].quantile(lower_quantile, dim='swath_num'),
                SWOT_ds_list[i]["KE_LLL"].quantile(upper_quantile, dim='swath_num'),
                color=LLL_color, alpha=0.2
            ) if LLL and 'LLL_mean' in SWOT_ds_list[i].data_vars else None
            axes[axes_val_swot].fill_between(
                SWOT_ds_list[i]["Kvar_Bessel"],
                SWOT_ds_list[i]["KE_Bessel"].quantile(lower_quantile, dim='swath_num'),
                SWOT_ds_list[i]["KE_Bessel"].quantile(upper_quantile, dim='swath_num'),
                color=Bessel_color, alpha=0.2
            ) if Bessels and 'KE_Bessel' in SWOT_ds_list[i].data_vars else None
    # Add text with full region name and mean latitude in the top left corner of the first column
    axes[0].text(0.05, 0.95, f"{params['Full name']} {params['Mean Latitude']}", transform=axes[0].transAxes,
                    fontsize=10, verticalalignment='top', fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    
    # Add text with dataset name and date range in the bottom left corner of each panel

    # define season_str based on season list, e.g. [12, 1, 2] -> DJF, [3, 4, 5] -> MAM, etc. Numbers like [2,5,7] that don't correspond to a standard season should just be joined with commas, e.g. "Feb, May, Jul"
    if season is not None:
        if season == [12, 1, 2]:
            season_str = "DJF"
        elif season == [3, 4, 5]:
            season_str = "MAM"
        elif season == [6, 7, 8]:
            season_str = "JJA"
        elif season == [9, 10, 11]:
            season_str = "SON"
        else:
            month_str_dict = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
            season_str = ", ".join([month_str_dict[month] for month in season])
            
    else:
        season_str = ""

    LLC4320_years = f"{ds_LLC4320['time'].min(skipna=True).dt.strftime('%Y').item()}" if ds_LLC4320['time'].min(skipna=True).dt.strftime('%Y').item() == ds_LLC4320['time'].max(skipna=True).dt.strftime('%Y').item() else f"{ds_LLC4320['time'].min(skipna=True).dt.strftime('%Y').item()} - {ds_LLC4320['time'].max(skipna=True).dt.strftime('%Y').item()}"

    simSWOT_years = []
    for ds in simSWOT_ds_list:
        simSWOT_years.append(f"{ds['time_coord'].min(skipna=True).dt.strftime('%Y').item()}" if ds['time_coord'].min(skipna=True).dt.strftime('%Y').item() == ds['time_coord'].max(skipna=True).dt.strftime('%Y').item() else f"{ds['time_coord'].min(skipna=True).dt.strftime('%Y').item()} - {ds['time_coord'].max(skipna=True).dt.strftime('%Y').item()}")
    SWOT_years = []
    for ds in SWOT_ds_list:
        SWOT_years.append(f"{ds['time_coord'].min(skipna=True).dt.strftime('%Y').item()}" if ds['time_coord'].min(skipna=True).dt.strftime('%Y').item() == ds['time_coord'].max(skipna=True).dt.strftime('%Y').item() else f"{ds['time_coord'].min(skipna=True).dt.strftime('%Y').item()} - {ds['time_coord'].max(skipna=True).dt.strftime('%Y').item()}")

    axes[0].text(0.05, 0.05, f"LLC4320 - {ds_LLC4320.time.size} snapshots \n {season_str} {LLC4320_years}", transform=axes[0].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    
    for i in range(len(simSWOT_ds_list)):
        axes_val_simSWOT = 1 if i == 0 else 2
        simSWOT_name = "SimSWOT Fast Phase" if i == 0 and phase == "both" else "SimSWOT Science Phase" if i == 1 and phase == "both" else "SimSWOT Science Phase" if phase == "science" else "SimSWOT Fast Phase"
        axes[axes_val_simSWOT].text(0.05, 0.05, f"{simSWOT_name} - {simSWOT_ds_list[i].swath_num.size} swaths\n{season_str} {simSWOT_years[i]}", transform=axes[axes_val_simSWOT].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    for i in range(len(SWOT_ds_list)):
        axes_val_SWOT = axes_val_simSWOT + 1 if i == 0 else axes_val_simSWOT + 2
        SWOT_name = "SWOT Fast Phase" if i == 0 and phase == "both" else "SWOT Science Phase" if i == 1 and phase == "both" else "SWOT Science Phase" if phase == "science" else "SWOT Fast Phase"
        axes[axes_val_SWOT].text(0.05, 0.05, f"{SWOT_name} - {SWOT_ds_list[i].swath_num.size} swaths\n{season_str} {SWOT_years[i]}", transform=axes[axes_val_SWOT].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))

    # Set ylim/ymax to be the same for all panels in a row based on the ylim/ymax of LLC4320 only

    # get number of columns to plot based on number of simSWOT and SWOT datasets
    colnum = 1 + len(simSWOT_ds_list) + len(SWOT_ds_list)
    for i in range(colnum):
        if any(v is not None for v in [KE_ymin, KE_ymax]):
            axes[i].set_ylim(KE_ymin, KE_ymax)
        # elif any(v is not None for v in [Q_ymin, Q_ymax]) and 'Enstrophy flux' in plotting_variable:
        #     axes[i].set_ylim(Q_ymin, Q_ymax)

        elif zoom:
            if zoom == "CG":
                # set ylim based on the max abs val EFlux_CG mean of the SWOT datasets
                num_SWOT_datasets = len(SWOT_ds_list)
                ax_limiter = 2 if (num_SWOT_datasets == 1) else 3  # use the second SWOT panel if there are two SWOT datasets, otherwise use the first SWOT panel
                axes[i].set_ylim(-5 * np.mean(np.abs(SWOT_ds_list[num_SWOT_datasets-1]['KE_CG'].mean(dim='swath_num'))), 5 * np.mean(np.abs(SWOT_ds_list[num_SWOT_datasets-1]['KE_CG'].mean(dim='swath_num'))))
            if zoom == "ASF":
                # set ylim based on the max abs val of the ASF mean of the SWOT datasets
                num_SWOT_datasets = len(SWOT_ds_list)
                ax_limiter = 2 if (num_SWOT_datasets == 1) else 3 # use the second SWOT panel if there are two SWOT datasets, otherwise use the first SWOT panel
                axes[i].set_ylim(-1.5 * np.mean(np.abs(SWOT_ds_list[num_SWOT_datasets-1]['KE_asf'].mean(dim='swath_num'))), 1.5 * np.mean(np.abs(SWOT_ds_list[num_SWOT_datasets-1]['KE_asf'].mean(dim='swath_num'))))
        else:
            # set ylim based on the max abs val of the SWOT ylim
            ax_limiter = 2 if (len(SWOT_ds_list) == 1) else 3 # use the second SWOT panel if there are two SWOT datasets, otherwise use the first SWOT panel
            axes[i].set_ylim(-1.1 * np.max(np.abs(axes[ax_limiter].get_ylim())), 1.1 * np.max(np.abs(axes[ax_limiter].get_ylim())))

        # if i == 0:
        #     # if convert_Wm3:
        #     #     plotting_variable = plotting_variable + r" [W m$^{-3}$]"
        #     # else:
        #     #     plotting_variable = plotting_variable + r" [m$^2$ s$^{-3}$]"

        # elif 'Enstrophy flux' in plotting_variable and i == 0:
        #     plotting_variable = plotting_variable + r" [s$^{-3}$]"
        axes[0].set_ylabel(r"Kinetic energy flux [m$^2$ s$^{-3}$]") if convert_KEFlux_to_wattperm3 is False else axes[0].set_ylabel(r"Kinetic energy flux [W m$^{-3}$]")
        axes[i].set_xlabel(r"Wavenumber [m$^{-1}$]")
        axes[i].set_xlim(1e-6, 1e-2)  # Set x-limits for all axes
        axes[i].hlines(0, 1e-6, 1e-2, colors="k", lw=1, zorder=0)
        secax = axes[i].secondary_xaxis('top', functions=(wavenumber_to_distance, distance_to_wavenumber))
        secax.set_xlabel("Separation Distance [m]")
        secax.set_xlim(wavenumber_to_distance(1e-2), wavenumber_to_distance(1e-6))  # Match limits to primary x-axis
        secax.tick_params(direction="in", which="both", bottom=False)

        if use_km:
            axes[i].set_xlabel(r"Wavenumber [km$^{-1}$]")
            secax.set_xlabel("Separation Distance [km]")
            axes[i].set_xlim(1e-3, 1e0)  # Set x-limits for all axes in km
            secax.set_xlim(wavenumber_to_distance(1e1), wavenumber_to_distance(1e-3))  # Match limits to primary x-axis in km
            axes[i].hlines(0, 1e-2, 1e0, colors="k", lw=1, zorder=0)

    axes[colnum-1].legend(loc='upper right')
        
    ds_LLC4320.close()
    for ds in simSWOT_ds_list:
        ds.close()
    for ds in SWOT_ds_list:
        ds.close()

### Plotting

In [ ]:
# Run plot_1x3_allmethods and iterate over regions in region_dict

for i in range(0, 7):
    region_name = list(region_dict.keys())[i]
    # if region_name in ["labradorsea", "westatlantic"]:
    #     continue
    # season_list = [None, [12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
    season_list = [None]
    # season_list = [[9, 10, 11]] # swot fast phase date range
    # try:
    for season in season_list:
        if season is None:
            season_str = "yearly"
        # set season_str to DJF, etc 
        if season == [12, 1, 2]:
            season_str = "DJF"
        elif season == [3, 4, 5]:
            season_str = "MAM"
        elif season == [6, 7, 8]:
            season_str = "JJA"
        elif season == [9, 10, 11]:
            season_str = "SON"
            
        plot_1x3_allmethods(region_dict, region_idx=i, confint=False, CG=True, Bessels=False, Bessel_mean_bug=True, ASF=True, LLL=False, KE=True, KE_ymin=None, KE_ymax=None, use_km=False, convert_KEFlux_to_wattperm3=False, season=season, phase="both",asf_dir="down",LLL_dir="down",scale=1, zoom="CG")
        # plt.show()
        plt.savefig(f"../figures/2026-06-03_LLC-simSWOT-SWOT_fast-science/KE_flux_ASF-CG_fast-science_noConfint_CGzoom_region_{region_name.replace(' ', '_')}_season_{season_str}.png", dpi=300)
# except Exception as e:
#     print(f"Error occurred while plotting for region {'acc'}: {e}")

### Single region plot

In [ ]:
for season in [[12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11], None]:

    plot_1x3_allmethods(region_dict, region_idx=2, confint=False, LLL=True, KE=True, KE_ymin=None, KE_ymax=None, Bessel_mean_bug=False, filter_outliers=False, use_km=False, convert_KEFlux_to_wattperm3=False, season=season, simSWOT_fastphase=True)
    plt.tight_layout()
    plt.savefig(f"figs/2026-04-27_LLC-simSWOT_fast-SWOT_science/KE_flux_ASF-CG-LLL_noConfint_region_{list(region_dict.keys())[2].replace(' ', '_')}_months_{season[0] if season is not None else 'all'}-{season[-1] if season is not None else 'all'}.png", dpi=300)
    # plt.show()

### Testing

In [ ]:
region_acc, params_acc = list(region_dict.items())[0]

# ds_LLC4320 = load_dataset("LLC4320_file", params_acc, region_acc)
# ds_simSWOT = load_dataset("simSWOT_file", params_acc, region_acc)
ds_SWOT = load_dataset("SWOT_file", params_acc, region_acc)

In [ ]:
# iterate over all files in directory
filedir = "/Users/cassswagner/OSU_research/SWOT_paper1/data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/science_phase/acc"
sorted_files = sorted(os.listdir(filedir))
for filename in sorted_files:
    # if filename starts with 2026 and ends with .nc, print filename
    if filename.startswith("2026") and filename.endswith(".nc"):
        ds_tmp = xr.open_dataset(os.path.join(filedir, filename))
        print(f"Coordinates in {filename}: {list(ds_tmp.coords)}")
        ds_tmp.close()

In [ ]:
ds_SWOT_1cycle

In [ ]:
ds_LLC4320.EFlux_CG.mean('time').plot()
ds_simSWOT.EFlux_CG.mean('swath_num').plot()
ds_SWOT.EFlux_CG.mean('swath_num').plot()
plt.xscale('log')
plt.legend(['LLC4320', 'simSWOT', 'SWOT'])
plt.show()

## 2026-06-03 overlay datasets one plot

### Functions

In [6]:
MONTHS = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}


def season_label(months):
    if months is None:
        return ""
    months = list(months)
    standard = {
        (12, 1, 2): "DJF",
        (3, 4, 5): "MAM",
        (6, 7, 8): "JJA",
        (9, 10, 11): "SON",
    }
    if tuple(months) in standard:
        return standard[tuple(months)]

    groups = []
    group = [months[0]]
    for month in months[1:]:
        if month == group[-1] + 1 or (group[-1] == 12 and month == 1):
            group.append(month)
        else:
            groups.append(group)
            group = [month]
    groups.append(group)

    parts = [
        MONTHS[g[0]] if len(g) == 1 else f"{MONTHS[g[0]]}-{MONTHS[g[-1]]}"
        for g in groups
    ]
    return ", ".join(parts)


def with_time_coord(ds):
    return ds.assign_coords(time_coord=ds["time"].mean(["num_lines", "num_pixels"]))


def set_ke(ds, asf_name, lll_name, diff_name, use_lll):
    if ds is None:
        return None
    
    ds["KE_asf"] = -0.5 * ds[asf_name]
    ds["KE_CG"] = ds["EFlux_CG"]
    ds["KE_LLL"] = (
        -(2 / (3 * ds[diff_name])) * ds[lll_name]
        if use_lll and lll_name in ds.data_vars
        else None
    )
    bessel_vars = [ds[var] for var in ds.data_vars if "Bessel_asf" in var]
    ds["KE_Bessel"] = sum(bessel_vars) / 4 if bessel_vars else None
    return ds


def date_bounds_text(ds, coord, label, season_txt=""):
    start = ds[coord].min(skipna=True)
    end = ds[coord].max(skipna=True)
    sy = start.dt.strftime("%Y").item()
    ey = end.dt.strftime("%Y").item()
    sm = start.dt.strftime("%b").item()
    em = end.dt.strftime("%b").item()
    if season_txt:
        return f"{label}: {season_txt} {sy}" if sy == ey else f"{label} - {season_txt} {sy}-{ey}"
    return f"{label} - {sm} {sy} to {em} {ey}"


def plot_overlay_datasets(
    region_dict,
    region_idx=0,
    confint=True,
    ASF=True,
    CG=True,
    Bessels=False,
    LLL=False,
    KE=True,
    KE_ymin=None,
    KE_ymax=None,
    filter_outliers=False,
    use_km=False,
    convert_KEFlux_to_wattperm3=False,
    season=None,
    phase="both",
    LLC=True,
    simSWOT=True,
    asf_dir="mean",
    LLL_dir="mean",
    scale=1,
    zoom=None,
):
    region, params = list(region_dict.items())[region_idx]
    season_txt = season_label(season) if season is not None else ""

    llc = None
    if LLC:
        llc = load_dataset("LLC4320_file", params, region)
        llc = llc.assign_coords(
            time=pd.date_range(start="2011-09-13", periods=llc.sizes["time"], freq="h")
        )

    phase_files = {
        "fast": ("simSWOT_fast_phase_file", "SWOT_fast_phase_file"),
        "science": ("simSWOT_file", "SWOT_file"),
    }
    entries = []

    for phase_name in ("fast", "science"):
        if phase not in ("both", phase_name):
            continue

        sim_key, swot_key = phase_files[phase_name]

        if simSWOT:
            entries.append(
                {
                    "kind": "sim",
                    "phase": phase_name,
                    "label": f"SimSWOT {phase_name.title()} Phase",
                    "color": 1 if phase_name == "fast" else 2,
                    "ds": with_time_coord(load_dataset(sim_key, params, region)),
                }
            )

        try:
            swot_ds = with_time_coord(load_dataset(swot_key, params, region))
        except Exception as exc:
            print(
                f"Warning: Could not assign time_coord to {phase_name} phase SWOT dataset for region {region} due to error: {exc}. Skipping this dataset."
            )
            swot_ds = None

        if swot_ds is not None:
            entries.append(
                {
                    "kind": "swot",
                    "phase": phase_name,
                    "label": f"SWOT {phase_name.title()} Phase",
                    "color": 3 if phase_name == "fast" else 4,
                    "ds": swot_ds,
                }
            )

    if season is not None:
        if llc is not None:
            llc = llc.sel(time=llc["time"].dt.month.isin(season))
        for entry in entries:
            entry["ds"] = entry["ds"].sel(
                time_coord=entry["ds"]["time_coord"].dt.month.isin(season)
            )
        if llc is not None and len(llc["time"]) == 0:
            print(
                f"Warning: No data available for LLC4320 in region {region} for season {season_txt}. Setting LLC4320 dataset to None for this plot."
            )
            llc = None
        for entry in entries:
            if len(entry["ds"]["time_coord"]) == 0:
                print(
                    f"Warning: No data available for {entry['label']} in region {region} for season {season_txt}. Setting this dataset to None for this plot."
                )
                entry["ds"] = None

    if KE:
        if llc is not None:
            set_ke(llc, "asf_mean", "LLL_mean", "i_diffs", LLL)
        for entry in entries:
            set_ke(
                entry["ds"],
                f"asf_{asf_dir}",
                f"LLL_{LLL_dir}",
                "num_lines_diffs",
                LLL,
            )

    asf_colorlist = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
    asf_linewidth, asf_linestyle = 2, "solid"
    CG_linewidth, CG_linestyle = 3, "dashed"
    Bessel_linewidth, Bessel_linestyle = 2, "dashdot"
    LLL_linewidth, LLL_linestyle = 2, "dotted"

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    if llc is not None:
        if CG:
            ax.semilogx(
                llc["L"] * 1333,
                llc["KE_CG"].mean("time"),
                c=asf_colorlist[0],
                ls=CG_linestyle,
                lw=CG_linewidth,
            )
        if ASF:
            ax.semilogx(
                llc["i_diffs"],
                llc["KE_asf"].mean("time"),
                c=asf_colorlist[0],
                ls=asf_linestyle,
                lw=asf_linewidth,
            )
        if LLL and "KE_LLL" in llc.data_vars:
            ax.semilogx(
                llc["i_diffs"],
                llc["KE_LLL"].mean("time"),
                c=asf_colorlist[0],
                ls=LLL_linestyle,
                lw=LLL_linewidth,
            )
        if Bessels and "KE_Bessel" in llc.data_vars:
            ax.semilogx(
                2 / llc["K"],
                llc["KE_Bessel"].mean("time"),
                c=asf_colorlist[0],
                ls=Bessel_linestyle,
                lw=Bessel_linewidth,
            )

    for entry in entries:
        if entry["ds"] is None:
            continue
        ds = entry["ds"]
        kind = entry["kind"]
        color = asf_colorlist[entry["color"]]
        dim = "time" if kind == "llc" else "swath_num"
        x_mult = 1333 if kind == "llc" else 2e3

        if CG:
            ax.semilogx(
                ds["L"] * x_mult,
                ds["KE_CG"].mean(dim),
                c=color,
                ls=CG_linestyle,
                lw=CG_linewidth,
            )
        if ASF:
            ax.semilogx(
                ds["num_lines_diffs"],
                ds["KE_asf"].mean(dim),
                c=color,
                ls=asf_linestyle,
                lw=asf_linewidth,
            )
        if LLL and "KE_LLL" in ds.data_vars:
            ax.semilogx(
                ds["num_lines_diffs"],
                ds["KE_LLL"].mean(dim),
                c=color,
                ls=LLL_linestyle,
                lw=LLL_linewidth,
            )
        if Bessels and "KE_Bessel" in ds.data_vars:
            ax.semilogx(
                2 / ds["K"],
                ds["KE_Bessel"].mean(dim),
                c=color,
                ls=Bessel_linestyle,
                lw=Bessel_linewidth,
            )

    ax.set_ylabel(
        r"Kinetic energy flux [W m$^{-3}$]"
        if convert_KEFlux_to_wattperm3
        else r"Kinetic energy flux [m$^2$ s$^{-3}$]"
    )
    ax.set_xlabel("Separation Distance [km]" if use_km else "Separation Distance [m]")
    if use_km:
        ax.set_xlim(1e0, 1e3)
    else:
        ax.set_xlim(1e3, 1e6)

    if KE_ymin is not None and KE_ymax is not None:
        ax.set_ylim(KE_ymin, KE_ymax)
    else:
        ds_list = ([llc] if llc is not None else []) + [entry["ds"] for entry in entries]
        cg_vals = [
            np.abs(ds["KE_CG"].mean("time" if "time" in ds.dims else "swath_num"))
            .mean()
            .item()
            for ds in ds_list
            if "KE_CG" in ds.data_vars
        ]
        asf_vals = [
            np.abs(ds["KE_asf"].mean("time" if "time" in ds.dims else "swath_num"))
            .mean()
            .item()
            for ds in ds_list
            if "KE_asf" in ds.data_vars
        ]
        mean_ke_cg = np.mean(cg_vals) if cg_vals else 0
        mean_ke_asf = np.mean(asf_vals) if asf_vals else 0
        mean_ke = max(mean_ke_cg, mean_ke_asf) if (CG and ASF) else mean_ke_cg if CG else mean_ke_asf
        ax.set_ylim(-5 * mean_ke, 5 * mean_ke)

    ax.hlines(0, 1e-2 if use_km else 1e3, 1e0 if use_km else 1e6, colors="k", lw=1, zorder=0)
    ax.tick_params(direction="in", which="both", bottom=False)

    handles = []
    if llc is not None:
        handles.append(
            plt.Line2D(
                [0],
                [0],
                color=asf_colorlist[0],
                lw=asf_linewidth,
                ls=asf_linestyle,
                label=f"LLC4320 [{llc.sizes['time']} snapshots]",
            )
        )

    for entry in entries:
        if entry["ds"] is None:
            continue
        handles.append(
            plt.Line2D(
                [0],
                [0],
                color=asf_colorlist[entry["color"]],
                lw=asf_linewidth,
                ls=asf_linestyle,
                label=f"{entry['label']} [{entry['ds'].sizes['swath_num']} swaths]",
            )
        )

    for label, enabled, lw, ls in (
        ("ASF", ASF, asf_linewidth, asf_linestyle),
        ("CG", CG, CG_linewidth, CG_linestyle),
        ("LLL", LLL, LLL_linewidth, LLL_linestyle),
        ("Bessel", Bessels, Bessel_linewidth, Bessel_linestyle),
    ):
        if enabled:
            handles.append(plt.Line2D([0], [0], color="k", lw=lw, ls=ls, label=label))

    ax.legend(handles=handles, loc="upper left")

    text_lines = []
    if llc is not None:
        text_lines.append(date_bounds_text(llc, "time", "LLC4320", season_txt))
    text_lines.extend(
        date_bounds_text(entry["ds"], "time_coord", entry["label"], season_txt)
        for entry in entries
        if entry["ds"] is not None
    )
    ax.text(
        0.025,
        0.025,
        "\n".join(text_lines),
        transform=ax.transAxes,
        fontsize=8,
        verticalalignment="bottom",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="k", alpha=0.8),
    )

    ax.set_title(f"{params['Full name']} {params['Mean Latitude']}")

    for ds in ([llc] if llc is not None else []) + [entry["ds"] for entry in entries]:
        if ds is not None:
            ds.close()

### Plotting

In [7]:
# Run plot_1x3_allmethods and iterate over regions in region_dict
sns.set_context("paper")

KE_ylim_list = [3e-8, 5e-8, 3e-8, 3e-7, 1e-7, 4e-7, 1e-7] # in order of acc, nwpacific, capebasin, nwaustralia, westatlantic, newcaledonia, labradorsea

# Put only the combinations you want to run here.
# Boolean flags are included in the filename only when they are True.
run_specs = [
    # {"CG": True, "Bessels": True, "ASF": True, "LLL": True, "phase": "both", "LLC": True, "simSWOT": True, "confint": False}, # all both phases
    # {"CG": True, "Bessels": True, "ASF": True, "LLL": True, "phase": "science", "LLC": True, "simSWOT": True, "confint": False}, # all science phase
    # {"CG": True, "Bessels": True, "ASF": True, "LLL": True, "phase": "fast", "LLC": True, "simSWOT": True, "confint": False}, # all fast phase

    # {"CG": True, "Bessels": False, "ASF": True, "LLL": False, "phase": "both", "LLC": True, "simSWOT": True, "confint": False}, # ASF and CG only both phases
    # {"CG": True, "Bessels": False, "ASF": True, "LLL": False, "phase": "science", "LLC": True, "simSWOT": True, "confint": False}, # ASF and CG only science phase
    # {"CG": True, "Bessels": False, "ASF": True, "LLL": False, "phase": "fast", "LLC": True, "simSWOT": True, "confint": False}, # ASF and CG only fast phase

    # # {"CG": False, "Bessels": False, "ASF": True, "LLL": True, "phase": "both", "LLC": True, "simSWOT": True, "confint": False}, # ASF and LLL only both phases
    # # {"CG": False, "Bessels": False, "ASF": True, "LLL": True, "phase": "science", "LLC": True, "simSWOT": True, "confint": False}, # ASF and LLL only science phase
    # # {"CG": False, "Bessels": False, "ASF": True, "LLL": True, "phase": "fast", "LLC": True, "simSWOT": True, "confint": False}, # ASF and LLL only fast phase

    # {"CG": True, "Bessels": False, "ASF": False, "LLL": False, "phase": "both", "LLC": True, "simSWOT": True, "confint": False}, # CG only both phases
    # {"CG": True, "Bessels": False, "ASF": False, "LLL": False, "phase": "science", "LLC": True, "simSWOT": True,("CG"): True}, # CG only science phase
    # {"CG": True, "Bessels": False, "ASF": False, "LLL": False, "phase": "fast", "LLC": True, "simSWOT": True, "confint": False}, # CG only fast phase

    # {"CG": True, "Bessels": False, "ASF": True, "LLL": False, "phase": "science", "LLC": False, "simSWOT": False, "confint": False}, # no LLC or simSWOT science phase
    # {"CG": True, "Bessels": False, "ASF": True, "LLL": False, "phase": "fast", "LLC": True, "simSWOT": False, "confint": False}, # no simSWOT science phase

    # specific needed plots for paper
    # {"CG": True, "Bessels": False, "ASF": True, "LLL": False, "phase": "both", "LLC": False, "simSWOT": False, "confint": False}, # ASF and CG only both phases no LLC or simSWOT
    # {"CG": True, "Bessels": False, "ASF": False, "LLL": False, "phase": "both", "LLC": True, "simSWOT": True, "confint": False}, # CG only both phases
    {"CG": True, "Bessels": False, "ASF": True, "LLL": False, "phase": "science", "LLC": True, "simSWOT": True, "confint": True}, # ASF and CG only science phase with confint
    {"CG": True, "Bessels": False, "ASF": False, "LLL": False, "phase": "science", "LLC": True, "simSWOT": True, "confint": True}, # CG only science phase with confint
    {"CG": False, "Bessels": False, "ASF": True, "LLL": False, "phase": "science", "LLC": True, "simSWOT": True, "confint": True}, # ASF only science phase with confint    
]

season_labels = {
    None: "yearly",
    (12, 1, 2): "DJF",
    (3, 4, 5): "MAM",
    (6, 7, 8): "JJA",
    (9, 10, 11): "SON",
}

region_names = list(region_dict.keys())
# season_list = [None, [12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
season_list = [None]
output_dir = "../figures/2026-06-03_LLC-simSWOT-SWOT_fast-science"

for i, region_name in enumerate(region_names[:7]):
    full_output_dir = f"{output_dir}/{region_name.replace(' ', '_')}"
    os.makedirs(full_output_dir, exist_ok=True)
    for season in season_list:
        season_key = None if season is None else tuple(season)
        season_str = season_labels[season_key]

        for run_kwargs in run_specs:
            figname_parts = [region_name, season_str]
            for key, value in run_kwargs.items():
                if isinstance(value, bool):
                    if value:
                        figname_parts.append(key)
                else:
                    figname_parts.append(f"{key}-{value}")
            figname = "_".join(figname_parts)

            print(f"Plotting {figname}...")
            plot_overlay_datasets(
                region_dict,
                region_idx=i,
                KE=True,
                KE_ymin=-KE_ylim_list[i],
                KE_ymax=KE_ylim_list[i],
                use_km=False,
                convert_KEFlux_to_wattperm3=False,
                season=season,
                asf_dir="mean",
                LLL_dir="mean",
                scale=1,
                zoom="CG",
                **run_kwargs,
            )
            plt.savefig(f"{full_output_dir}/{figname}.png", dpi=300)
            plt.close()


Plotting acc_yearly_CG_ASF_phase-science_LLC_simSWOT_confint...
Plotting acc_yearly_CG_phase-science_LLC_simSWOT_confint...
Plotting acc_yearly_ASF_phase-science_LLC_simSWOT_confint...
Plotting nwpacific_yearly_CG_ASF_phase-science_LLC_simSWOT_confint...
Plotting nwpacific_yearly_CG_phase-science_LLC_simSWOT_confint...
Plotting nwpacific_yearly_ASF_phase-science_LLC_simSWOT_confint...
Plotting capebasin_yearly_CG_ASF_phase-science_LLC_simSWOT_confint...
Plotting capebasin_yearly_CG_phase-science_LLC_simSWOT_confint...
Plotting capebasin_yearly_ASF_phase-science_LLC_simSWOT_confint...
Plotting nwaustralia_yearly_CG_ASF_phase-science_LLC_simSWOT_confint...
Plotting nwaustralia_yearly_CG_phase-science_LLC_simSWOT_confint...
Plotting nwaustralia_yearly_ASF_phase-science_LLC_simSWOT_confint...
Plotting westatlantic_yearly_CG_ASF_phase-science_LLC_simSWOT_confint...
Plotting westatlantic_yearly_CG_phase-science_LLC_simSWOT_confint...
Plotting westatlantic_yearly_ASF_phase-science_LLC_simSWOT

## 2026-06-05 SSH LLC snapshots

In [ ]:
# add original LLC4320 snapshots to region_dict for each region
region_dict["acc"]['LLC4320_snapshot'] = "/Volumes/Promise Disk/data/mitgcm/acc_data/acc_all/LLC4320_pre-SWOT_ACC_SMST_20110913.nc"
region_dict["nwpacific"]['LLC4320_snapshot'] = "/Volumes/Promise Disk/data/mitgcm/nwpacific_data/nwpacific_all/LLC4320_pre-SWOT_NWPacific_20110913.nc"
region_dict["capebasin"]['LLC4320_snapshot'] = "/Volumes/Promise Disk/data/mitgcm/capebasin_data/capebasin_all/LLC4320_pre-SWOT_CapeBasin_20110913.nc"
region_dict["nwaustralia"]['LLC4320_snapshot'] = "/Volumes/Promise Disk/data/mitgcm/nwaustralia_data/nwaustralia_all/LLC4320_pre-SWOT_NWAustralia_20110913.nc"
region_dict["westatlantic"]['LLC4320_snapshot'] = "/Volumes/Promise Disk/data/mitgcm/westatlantic_data/westatlantic_all/LLC4320_pre-SWOT_WestAtlantic_20110913.nc"
region_dict["newcaledonia"]['LLC4320_snapshot'] = "/Volumes/Promise Disk/data/mitgcm/newcaledonia_data/newcaledonia_all/LLC4320_pre-SWOT_NewCaledonia_20110913.nc"
region_dict["labradorsea"]['LLC4320_snapshot'] = "/Volumes/Promise Disk/data/mitgcm/labradorsea_data/labradorsea_all/LLC4320_pre-SWOT_LabradorSea_20110913.nc"

for i in range(0, 7):
    region_name = list(region_dict.keys())[i]
    
    ds = xr.open_dataset(region_dict[region_name]['LLC4320_snapshot'])
    plt.pcolormesh(ds['XC'], ds['YC'], ds['Eta'].isel(time=0), cmap='viridis')
    plt.colorbar(label='Sea Surface Height (m)')
    plt.title(f"LLC4320 Snapshot - {region_name}")
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()

### Animations for each region

In [ ]:
region_dict["acc"]['LLC4320_dir'] = "/Volumes/Promise Disk/data/mitgcm/acc_data/acc_all"
region_dict["nwpacific"]['LLC4320_dir'] = "/Volumes/Promise Disk/data/mitgcm/nwpacific_data/nwpacific_all"
region_dict["capebasin"]['LLC4320_dir'] = "/Volumes/Promise Disk/data/mitgcm/capebasin_data/capebasin_all"
region_dict["nwaustralia"]['LLC4320_dir'] = "/Volumes/Promise Disk/data/mitgcm/nwaustralia_data/nwaustralia_all"
region_dict["westatlantic"]['LLC4320_dir'] = "/Volumes/Promise Disk/data/mitgcm/westatlantic_data/westatlantic_all"
region_dict["newcaledonia"]['LLC4320_dir'] = "/Volumes/Promise Disk/data/mitgcm/newcaledonia_data/newcaledonia_all"
region_dict["labradorsea"]['LLC4320_dir'] = "/Volumes/Promise Disk/data/mitgcm/labradorsea_data/labradorsea_all"

sns.set_context("paper")
for i in range(0, 7):
    region_name = list(region_dict.keys())[i]
    full_name = region_dict[region_name]['Full name']
    files = sorted(glob.glob(os.path.join(region_dict[region_name]['LLC4320_dir'], "LLC4320_pre-SWOT_*.nc")))
    for file in files:
        ds = xr.open_dataset(file)
        date = ds['time'].isel(time=0).dt.strftime('%Y-%m-%d').item()
        for time in ds['time']:

            # check if image for this time already exists, and if so, skip to the next time
            output_path = f"../figures/2026-06-05_LLC4320_snapshots-animation/{region_name}"
            os.makedirs(output_path, exist_ok=True)
            output_file = f"{output_path}/{file.split('/')[-1].split('.')[0]}_time{time.values}.png"
            if os.path.exists(output_file):
                continue

            time_niceprint_hour = time.dt.strftime('%H').item()
            fig, ax = plt.subplots(figsize=(8, 6))
            plt.pcolormesh(ds['XC'], ds['YC'], ds['Eta'].sel(time=time), cmap='viridis', vmin=-1, vmax=1)
            plt.colorbar(label='Sea Surface Height (m)')
            plt.title(f"{full_name}: {date} - {time_niceprint_hour}:00")
            plt.xlabel('Longitude')
            plt.ylabel('Latitude')
            plt.tight_layout()
            plt.savefig(output_file, dpi=300)
            plt.close()
        ds.close()

## 2026-04-17 LLC4320-simSWOT LLL ACC

### Functions

In [ ]:
def plot_LLC_simSWOT_allmethods(region_dict, confint=True, KE=True, Bessel_mean_bug=True, use_km=False, convert_KEFlux_to_wattperm3=False, cut_to_LLC4320_daterange=False, KE_ymin=None, KE_ymax=None, llc_scale_factor=1):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

    region, params = list(region_dict.items())[0]

    ds_LLC4320 = load_dataset("LLC4320_file", params, region)
    ds_LLC4320_LLL = load_dataset("LLC4320_LLL_file", params, region)

    # merge ds_LLC4320 and ds_LLC4320_LLL on time coordinate, keeping all variables from both datasets and slicing ds_LLC4320_LLL to the same time range as ds_LLC4320
    ds_LLC4320_LLL = ds_LLC4320_LLL.isel(time=slice(ds_LLC4320.time.min().values, ds_LLC4320.time.max().values+1))
    ds_LLC4320 = xr.merge([ds_LLC4320, ds_LLC4320_LLL], compat='override')

    ds_simSWOT = load_dataset("simSWOT_file", params, region)
    ds_simSWOT_LLL = load_dataset("simSWOT_LLL_file", params, region)

    # merge simSWOT and simSWOT_LLL on swath_num and time coordinates, keeping all variables from both datasets
    ds_simSWOT = xr.merge([ds_simSWOT, ds_simSWOT_LLL], compat='override')

    if cut_to_LLC4320_daterange:

        # Add time coordinate to simSWOT from simSWOT variables
        ds_simSWOT = ds_simSWOT.assign_coords(time_coord=ds_simSWOT['time'].mean(["num_lines", "num_pixels"]))

        # Select only the time range of simSWOT that overlaps with LLC4320, but LLC4320.time is just an index, so use _format_date_range to change it to an actual time that can be used to slice ds_simSWOT
        date_range = _format_date_range(ds_LLC4320)
        start_date, end_date = date_range.split(" – ")
        start_ts = pd.to_datetime(start_date)
        end_ts = pd.to_datetime(end_date)

        time_coord = ds_simSWOT["time_coord"]
        time_mask = (time_coord >= start_ts) & (time_coord <= end_ts)

        # time is indexed by swath_num in this dataset
        ds_simSWOT = ds_simSWOT.isel(swath_num=time_mask)

    # Set flux variables
    if KE is True:

        ds_LLC4320_KE_asf = -0.5 * ds_LLC4320['asf_mean'] * llc_scale_factor
        ds_simSWOT_KE_asf = -0.5 * ds_simSWOT['asf_mean']

        # ds_LLC4320_enst_asf = 2 * ds_LLC4320['asf_mean'] * (1318 / 2000) ** 2 / ds_LLC4320['i_diffs']**2
        # ds_simSWOT_enst_asf = 2 * ds_simSWOT['asf_down'] / ds_simSWOT['num_lines_diffs']**2
        # ds_SWOT_enst_asf = 2 * ds_SWOT['asf_down'] / ds_SWOT['num_lines_diffs']**2

        ds_LLC4320_KE_CG = ds_LLC4320['EFlux_CG'] * llc_scale_factor
        ds_simSWOT_KE_CG = ds_simSWOT['EFlux_CG'] 

        ds_LLC4320_KE_LLL = - (2/(3 * ds_LLC4320.i_diffs)) * ds_LLC4320['LLL_mean'] * llc_scale_factor
        ds_simSWOT_KE_LLL = - (2/(3 * ds_simSWOT.num_lines_diffs)) * ds_simSWOT['LLL_mean']

        if Bessel_mean_bug:

            ds_LLC4320["EFlux_Bessel_ASF_mean_tapered"] = (sum(ds_LLC4320[var] for var in ds_LLC4320.data_vars if "Bessel_asf" in var)/4)
            ds_simSWOT["EFlux_Bessel_ASF_mean_tapered"] = (sum(ds_simSWOT[var] for var in ds_simSWOT.data_vars if "Bessel_asf" in var)/4)


        ds_LLC4320_KE_Bessel = ds_LLC4320['EFlux_Bessel_ASF_mean_tapered'] * llc_scale_factor
        ds_simSWOT_KE_Bessel = ds_simSWOT['EFlux_Bessel_ASF_mean_tapered'] 

        if convert_KEFlux_to_wattperm3:
            ds_LLC4320_KE_asf *= 1025
            ds_simSWOT_KE_asf *= 1025

            ds_LLC4320_KE_CG *= 1025
            ds_simSWOT_KE_CG *= 1025

            ds_LLC4320_KE_Bessel *= 1025
            ds_simSWOT_KE_Bessel *= 1025

            ds_LLC4320_KE_LLL *= 1025
            ds_simSWOT_KE_LLL *= 1025

    # Set x/K variables
    ds_LLC4320_Kvar_asf = 1 / ds_LLC4320['i_diffs']
    ds_simSWOT_Kvar_asf = 1 / ds_simSWOT['num_lines_diffs']

    print("Minimum i_diffs in LLC4320:", ds_LLC4320['i_diffs'][1].values)
    print("Minimum num_lines_diffs in simSWOT:", ds_simSWOT['num_lines_diffs'][1].values)

    ds_LLC4320_Kvar_CG = ds_LLC4320['K_coarse_grain'].mean('time') / (2*np.pi)
    ds_simSWOT_Kvar_CG = ds_simSWOT['K_coarse_grain'].mean('swath_num') / (2*np.pi)

    print("Minimum sep distance for CG in LLC4320:", 1/ds_LLC4320_Kvar_CG.max().values)
    print("Minimum sep distance for CG in simSWOT:", 1/ds_simSWOT_Kvar_CG.max().values)

    ds_LLC4320_Kvar_Bessel = ds_LLC4320['K']
    ds_simSWOT_Kvar_Bessel = ds_simSWOT['K']

    ds_LLC4320_Kvar_LLL = 1 / ds_LLC4320['i_diffs']
    ds_simSWOT_Kvar_LLL = 1 / ds_simSWOT['num_lines_diffs']

    if use_km:
        ds_LLC4320_Kvar_asf *= 1e3
        ds_simSWOT_Kvar_asf *= 1e3

        ds_LLC4320_Kvar_CG *= 1e3
        ds_simSWOT_Kvar_CG *= 1e3

        # ds_LLC4320_Kvar_Bessel *= 1e-3
        # ds_simSWOT_Kvar_Bessel *= 1e-3

        ds_LLC4320_Kvar_LLL *= 1e3
        ds_simSWOT_Kvar_LLL *= 1e3

    # Plotting parameters
    asf_color = 'tab:red'
    asf_linewidth = 2
    asf_linestyle = '--'

    CG_color = 'k'
    CG_linewidth = 3
    CG_linestyle = 'solid'

    Bessel_color = 'tab:blue'
    Bessel_linewidth = 2
    Bessel_linestyle = 'solid'

    LLL_color = 'tab:blue'
    LLL_linewidth = 2
    LLL_linestyle = 'solid'

    # Plot LLC4320 variables
    axes[0].semilogx(ds_LLC4320_Kvar_asf, ds_LLC4320_KE_asf.mean('time'),c=asf_color,ls=asf_linestyle, lw=asf_linewidth, label='ASF')
    axes[0].semilogx(ds_LLC4320_Kvar_CG, ds_LLC4320_KE_CG.mean('time'),c=CG_color,ls=CG_linestyle,lw=CG_linewidth, label='CG')
    # axes[0].semilogx(ds_LLC4320_Kvar_Bessel, ds_LLC4320_KE_Bessel.mean('time'),c=Bessel_color,ls=Bessel_linestyle, lw=Bessel_linewidth)
    axes[0].semilogx(ds_LLC4320_Kvar_LLL, ds_LLC4320_KE_LLL.mean('time'),c=LLL_color,ls=LLL_linestyle, lw=LLL_linewidth, label='LLL')

    # Plot simSWOT variables
    axes[1].semilogx(ds_simSWOT_Kvar_asf, ds_simSWOT_KE_asf.mean('swath_num'),c=asf_color,ls=asf_linestyle, lw=asf_linewidth, label='ASF')
    axes[1].semilogx(ds_simSWOT_Kvar_CG, ds_simSWOT_KE_CG.mean('swath_num'),c=CG_color,ls=CG_linestyle,lw=CG_linewidth, label='CG')
    # axes[1].semilogx(ds_simSWOT_Kvar_Bessel, ds_simSWOT_KE_Bessel.mean('swath_num'),c=Bessel_color,ls=Bessel_linestyle, lw=Bessel_linewidth)
    axes[1].semilogx(ds_simSWOT_Kvar_LLL, ds_simSWOT_KE_LLL.mean('swath_num'),c=LLL_color,ls=LLL_linestyle, lw=LLL_linewidth, label='LLL')

    if confint:
        upper_quantile = 0.90
        lower_quantile = 0.1

        # Plot LLC4320 confidence intervals
        axes[0].fill_between(
            ds_LLC4320_Kvar_asf, 
            ds_LLC4320_KE_asf.quantile(lower_quantile, dim='time'),
            ds_LLC4320_KE_asf.quantile(upper_quantile, dim='time'),
            color=asf_color, alpha=0.2
                             )
        axes[0].fill_between(
            ds_LLC4320_Kvar_CG, 
            ds_LLC4320_KE_CG.quantile(lower_quantile, dim='time'),
            ds_LLC4320_KE_CG.quantile(upper_quantile, dim='time'),
            color=CG_color, alpha=0.2
                             )
        # axes[0].fill_between(
        #     ds_LLC4320_Kvar_Bessel, 
        #     ds_LLC4320_KE_Bessel.quantile(lower_quantile, dim='time'),
        #     ds_LLC4320_KE_Bessel.quantile(upper_quantile, dim='time'),
        #     color=Bessel_color, alpha=0.2
        #                      )        

        axes[0].fill_between(
            ds_LLC4320_Kvar_LLL,
            ds_LLC4320_KE_LLL.quantile(lower_quantile, dim='time'),
            ds_LLC4320_KE_LLL.quantile(upper_quantile, dim='time'),
            color=LLL_color, alpha=0.2
        )


        # Plot simSWOT confidence intervals
        axes[1].fill_between(
            ds_simSWOT_Kvar_asf,
            ds_simSWOT_KE_asf.quantile(lower_quantile, dim='swath_num'),
            ds_simSWOT_KE_asf.quantile(upper_quantile, dim='swath_num'),
            color=asf_color, alpha=0.2
        )
        axes[1].fill_between(
            ds_simSWOT_Kvar_CG,
            ds_simSWOT_KE_CG.quantile(lower_quantile, dim='swath_num'),
            ds_simSWOT_KE_CG.quantile(upper_quantile, dim='swath_num'),
            color=CG_color, alpha=0.2
        )
        # axes[1].fill_between(
        #     ds_simSWOT_Kvar_Bessel,
        #     ds_simSWOT_KE_Bessel.quantile(lower_quantile, dim='swath_num'),
        #     ds_simSWOT_KE_Bessel.quantile(upper_quantile, dim='swath_num'),
        #     color=Bessel_color, alpha=0.2
        # )

        axes[1].fill_between(
            ds_simSWOT_Kvar_LLL,
            ds_simSWOT_KE_LLL.quantile(lower_quantile, dim='swath_num'),
            ds_simSWOT_KE_LLL.quantile(upper_quantile, dim='swath_num'),
            color=LLL_color, alpha=0.2
        )


    # Add text with full region name and mean latitude in the top left corner of the first column
    axes[0].text(0.05, 0.95, f"{params['Full name']} {params['Mean Latitude']}", transform=axes[0].transAxes,
                    fontsize=10, verticalalignment='top', fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    
    # Add text with dataset name and date range in the bottom left corner of each panel
    axes[0].text(0.05, 0.05, f"LLC4320\n{_format_date_range(ds_LLC4320)}", transform=axes[0].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    
    axes[1].text(0.05, 0.05, f"SimSWOT\n{_format_date_range(ds_simSWOT)}", transform=axes[1].transAxes,
                    fontsize=10, verticalalignment='bottom',
                    bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
    
    # Set ylim/ymax to be the same for all panels in a row based on the ylim/ymax of LLC4320 only
    for i in range(2):
        try:
            axes[i].set_ylim(KE_ymin, KE_ymax)
        except:
            pass
        axes[0].set_ylabel(r"Kinetic energy flux [m$^2$ s$^{-3}$]")
        axes[i].set_xlabel(r"Wavenumber [m$^{-1}$]")
        axes[i].set_xlim(1e-6, 1e-2)  # Set x-limits for all axes
        axes[i].hlines(0, 1e-6, 1e-2, colors="k", lw=1, zorder=0)
        secax = axes[i].secondary_xaxis('top', functions=(wavenumber_to_distance, distance_to_wavenumber))
        secax.set_xlabel("Separation Distance [m]")
        secax.set_xlim(wavenumber_to_distance(1e-2), wavenumber_to_distance(1e-6))  # Match limits to primary x-axis
        secax.tick_params(direction="in", which="both", bottom=False)

        if use_km:
            axes[i].set_xlabel(r"Wavenumber [km$^{-1}$]")
            secax.set_xlabel("Separation Distance [km]")
            axes[i].set_xlim(1e-3, 1e1)  # Set x-limits for all axes in km
            secax.set_xlim(wavenumber_to_distance(1e1), wavenumber_to_distance(1e-3))  # Match limits to primary x-axis in km
            axes[i].hlines(0, 1e-3, 1e1, colors="k", lw=1, zorder=0)

            # draw vertical line at 30km for acc region
            # axes[i].axvline(x=1/30, color='k', ls='--', lw=1) 

    if convert_KEFlux_to_wattperm3:
        axes[0].set_ylabel(r"Kinetic energy flux [W m$^{-3}$]")
        axes[1].set_ylabel(r"Kinetic energy flux [W m$^{-3}$]")

        # draw horizontal green line at -6.5e-6 for LLC4320 and horizontal green line at -5e-6 for simSWOT and black line at -3.5e-6 for simSWOT
        axes[0].hlines(-6.5e-6, 1e-3, 1e1, colors="green", lw=1, zorder=0, linestyle='--')
        axes[1].hlines(-5e-6, 1e-3, 1e1, colors="green", lw=1, zorder=0, linestyle='--', label="Wang 2025 FFT peak")
        axes[1].hlines(-3.5e-6, 1e-3, 1e1, colors="k", lw=1, zorder=0, label="Wang 2025 CG peak", linestyle='--')
    
    axes[1].legend(loc='lower right')
        
    plt.tight_layout()

### Plotting

In [ ]:
plot_LLC_simSWOT_allmethods(region_dict, confint=False, KE=True, Bessel_mean_bug=True, use_km=True, convert_KEFlux_to_wattperm3=True, cut_to_LLC4320_daterange=True, KE_ymin=-2e-5, KE_ymax=1e-5)
# plt.savefig(f"figs/2026-04-08_1x3_allmethod_allregion/KE_flux_allmethods_noBessel_noConfint_region_{region_name.replace(' ', '_')}.png", dpi=300)

### Functions

In [ ]:
# 5x3 panel plot to plot 5 regions with 3 datasets each, and a toggle between which SF/CG/etc to show
def plot_5x3_panel_dirs(region_dict, SWOT_filename=None, plotting_variable='KE flux from ASF', confint=True, color='tab:blue', linestyle='-', num_regions=5, ymin=None, ymax=None):
    fig, axes = plt.subplots(num_regions, 3, figsize=(15, 4 * num_regions), sharex=True)

    for i, (region, params) in enumerate(region_dict.items()):
        if i >= num_regions:
            break  # Only plot specified number of regions

        if SWOT_filename is not None:
            ds_SWOT = xr.open_dataset(SWOT_filename).mean(["num_lines","num_pixels"])
        else:
            ds_SWOT = load_dataset("SWOT_file", params, region).mean(["num_lines","num_pixels"])
        ds_SWOT_NE = ds_SWOT.where(ds_SWOT['direction'] == 'northeast', drop=True)
        ds_SWOT_NW = ds_SWOT.where(ds_SWOT['direction'] == 'northwest', drop=True)

        # If no datasets are available, skip to the next region
        if all(ds is None for ds in [ds_SWOT, ds_SWOT_NE, ds_SWOT_NW]):
            continue

        if plotting_variable == 'KE flux from ASF':

            ds_SWOT_var = -0.5 * ds_SWOT['asf_down']
            ds_SWOT_NE_var = -0.5 * ds_SWOT_NE['asf_down']
            ds_SWOT_NW_var = -0.5 * ds_SWOT_NW['asf_down']

            ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']
            ds_SWOT_NE_xvar = 2 * np.pi / ds_SWOT_NE['num_lines_diffs']
            ds_SWOT_NW_xvar = 2 * np.pi / ds_SWOT_NW['num_lines_diffs']

        if plotting_variable == 'Enstrophy flux from ASF':

            ds_SWOT_var = 2 * ds_SWOT['asf_down'] / ds_SWOT['num_lines_diffs']**2
            ds_SWOT_NE_var = 2 * ds_SWOT_NE['asf_down'] / ds_SWOT_NE['num_lines_diffs']**2
            ds_SWOT_NW_var = 2 * ds_SWOT_NW['asf_down'] / ds_SWOT_NW['num_lines_diffs']**2

            ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']
            ds_SWOT_NE_xvar = 2 * np.pi / ds_SWOT_NE['num_lines_diffs']
            ds_SWOT_NW_xvar = 2 * np.pi / ds_SWOT_NW['num_lines_diffs']

        if plotting_variable == 'Enstrophy flux from dwAw':

            ds_SWOT_var = -0.5 * ds_SWOT['asfq_down']
            ds_SWOT_NE_var = -0.5 * ds_SWOT_NE['asfq_down']
            ds_SWOT_NW_var = -0.5 * ds_SWOT_NW['asfq_down']

            ds_SWOT_xvar = 2 * np.pi / ds_SWOT['num_lines_diffs']
            ds_SWOT_NE_xvar = 2 * np.pi / ds_SWOT_NE['num_lines_diffs']
            ds_SWOT_NW_xvar = 2 * np.pi / ds_SWOT_NW['num_lines_diffs']

        if plotting_variable == 'KE flux from CG':

            ds_SWOT_var = ds_SWOT['EFlux_CG']
            ds_SWOT_NE_var = ds_SWOT_NE['EFlux_CG']
            ds_SWOT_NW_var = ds_SWOT_NW['EFlux_CG']

            ds_SWOT_xvar = ds_SWOT['K_coarse_grain'].mean('swath_num')
            ds_SWOT_NE_xvar = ds_SWOT_NE['K_coarse_grain'].mean('swath_num')
            ds_SWOT_NW_xvar = ds_SWOT_NW['K_coarse_grain'].mean('swath_num')
            

        if plotting_variable == 'Enstrophy flux from CG':

            ds_SWOT_var = ds_SWOT['QFlux_CG']
            ds_SWOT_NE_var = ds_SWOT_NE['QFlux_CG']
            ds_SWOT_NW_var = ds_SWOT_NW['QFlux_CG']

            ds_SWOT_xvar = ds_SWOT['K_coarse_grain'].mean('swath_num')
            ds_SWOT_NE_xvar = ds_SWOT_NE['K_coarse_grain'].mean('swath_num')
            ds_SWOT_NW_xvar = ds_SWOT_NW['K_coarse_grain'].mean('swath_num')

            
        if num_regions > 1:
            axes_i = axes[i]
        else:
            axes_i = axes
        axes_i[0].semilogx(
            ds_SWOT_xvar,
            ds_SWOT_var.mean('swath_num'),
            color=color,
            linestyle=linestyle,
            label='SWOT',
        )

        axes_i[1].semilogx(
            ds_SWOT_NE_xvar,
            ds_SWOT_NE_var.mean('swath_num'),
            color=color,
            linestyle=linestyle,
            label='SWOT NE',
        )

        axes_i[2].semilogx(
            ds_SWOT_NW_xvar,
            ds_SWOT_NW_var.mean('swath_num'),
            color=color,
            linestyle=linestyle,
            label='SWOT NW',
        )

        upper_quantile = 0.95
        lower_quantile = 0.05
        if confint:
            axes_i[0].fill_between(
                ds_SWOT_xvar,
                ds_SWOT_var.quantile(lower_quantile, dim='swath_num'),
                ds_SWOT_var.quantile(upper_quantile, dim='swath_num'),
                color=color, alpha=0.2
            )

            axes_i[1].fill_between(
                ds_SWOT_NE_xvar,
                ds_SWOT_NE_var.quantile(lower_quantile, dim='swath_num'),
                ds_SWOT_NE_var.quantile(upper_quantile, dim='swath_num'),
                color=color, alpha=0.2
            )

            axes_i[2].fill_between(
                ds_SWOT_NW_xvar,
                ds_SWOT_NW_var.quantile(lower_quantile, dim='swath_num'),
                ds_SWOT_NW_var.quantile(upper_quantile, dim='swath_num'),
                color=color, alpha=0.2
            )

        # Add text with full region name in the top left corner of the first column
        axes_i[0].text(0.05, 0.95, params["Full name"], transform=axes_i[0].transAxes,
                        fontsize=10, verticalalignment='top', fontweight='bold',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
        
        # Add text with dataset name in the bottom left corner of each panel
        axes_i[0].text(0.05, 0.05, 'SWOT', transform=axes_i[0].transAxes,
                        fontsize=10, verticalalignment='bottom',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
        axes_i[1].text(0.05, 0.05, 'SWOT NE', transform=axes_i[1].transAxes,
                        fontsize=10, verticalalignment='bottom',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))
        axes_i[2].text(0.05, 0.05, 'SWOT NW', transform=axes_i[2].transAxes,
                        fontsize=10, verticalalignment='bottom',
                        bbox=dict(boxstyle='round', facecolor='lightgrey', edgecolor='k', alpha=0.8))

        # Set ylim/ymax to be the same for all panels in a row based on the ylim/ymax of LLC4320 only



        for j in range(3):
            axes_i[j].set_ylim(axes_i[0].get_ylim())
            if ymin and ymax:
                axes_i[j].set_ylim(ymin, ymax)
            axes_i[j].set_ylabel(plotting_variable)
            axes_i[j].set_xlabel("Wavenumber [m$^{-1}$]")
            axes_i[j].set_xlim(1e-6, 1e-2)  # Set x-limits for all axes
            axes_i[j].hlines(0, 1e-6, 1e-2, colors="k", lw=1, zorder=0)
            secax = axes_i[j].secondary_xaxis('top', functions=(wavenumber_to_distance, distance_to_wavenumber))
            secax.set_xlabel("Separation Distance [m]")
            secax.set_xlim(wavenumber_to_distance(1e-2), wavenumber_to_distance(1e-6))  # Match limits to primary x-axis
            secax.tick_params(direction="in", which="both", bottom=False)
            
        ds_SWOT.close()
        ds_SWOT_NE.close()
        ds_SWOT_NW.close()
    plt.tight_layout()

### Plotting methods separately

In [ ]:
# Run the 5x3 panel plot function

sns.set_context("notebook")

var_dict = {
    'KE flux from ASF': ('tab:blue', '-'),
    'Enstrophy flux from ASF': ('tab:red', '-'),
    'Enstrophy flux from dwAw': ('tab:red', '-'),
    'KE flux from CG': ('tab:blue', '-'),
    'Enstrophy flux from CG': ('tab:red', '-'),
}

for plotting_variable, (color, linestyle) in var_dict.items():
    print(f"Plotting {plotting_variable}")
    plot_5x3_panel_dirs(region_dict_swot_dirs, plotting_variable=plotting_variable, color=color, linestyle=linestyle, confint=False, num_regions=1, ymin=-2e-7 if 'KE' in plotting_variable else -4e-15, ymax=4e-7 if 'KE' in plotting_variable else 2e-15)
    # plt.close('all')
    # plt.savefig(f"figs/{plotting_variable.replace(' ', '_')}_MITgcm-SimSWOT-SWOT_5x3_panel.png", dpi=300)

### Plotting methods together

In [ ]:
# add all regions except acc to skip regions
skip_regions = [region for region in region_dict.keys() if region != "acc"]

for region, params in region_dict.items():

    if region in skip_regions:
        continue

    ds_SWOT = load_dataset("SWOT_file", params, region).mean(["num_lines","num_pixels"])
    ds_SWOT_NE = ds_SWOT.where(ds_SWOT['direction'] == 'northeast', drop=True)
    ds_SWOT_NW = ds_SWOT.where(ds_SWOT['direction'] == 'northwest', drop=True)

    # If no datasets are available, skip to the next region
    if all(ds is None for ds in [ds_SWOT, ds_SWOT_NE, ds_SWOT_NW]):
        continue

    datasets = {
        "SWOT": ds_SWOT,
        "SWOT_NE": ds_SWOT_NE,
        "SWOT_NW": ds_SWOT_NW,
    }

    # Process datasets
    for key in datasets:

        datasets[key] = process_dataset(datasets[key], "asf_down", "asfq_down", 0.90, 0.10, "swath_num")

    # Create subplots
    fig, axes = plt.subplots(1, len(datasets), figsize=(5 * (len(datasets)+1), 5), sharex=True)
    if len(datasets) == 1:
        axes = [axes]  # Ensure axes is a list even for a single subplot
    quantile_alpha = 0.2
    paired = plt.cm.get_cmap("Paired")
    colors = [paired(i) for i in range(12)]
    
    
    blue, lightblue, red, lightred = colors[1], colors[1], colors[5], colors[5]
    ke_flux_label = r"$-\frac{1}{2}\delta u \delta \mathcal{A}_u$"
    enstrophy_flux_label = r"$\frac{2}{r^2} \delta u \delta \mathcal{A}_u$"
    dwdAw_label = r"$-\frac{1}{2}\delta \omega \delta A\omega$"
    LLL_label = r"$-\frac{2}{3r} \delta u_L \delta u_L \delta u_L$"
    Ens_LLL_label = r"$\frac{8}{r^3} \delta u_L \delta u_L \delta u_L$"
    Lww_label = r"$-\frac{1}{2r} \delta u_L \delta \omega \delta \omega$ "
    CG_label = "Coarse graining"
    CG_label_q = "Coarse graining"
    SF = True
    CG = True
    Ens = False
    KE = True
    dwAw = False
    LLL = True
    Ens_LLL = False
    Lww = False
    confint = False
    draw_Rossby_radius = False
    convert_KEFlux_to_wattperkm2permeter = False
    quantile_upper = 0.90
    quantile_lower = 0.10
    KE_lims_scale = 2e0
    Q_lims_scale = 5e1

    KE_ymin_override = -2e-7
    KE_ymax_override = 4e-7
    Q_ymin_override = -4e-15
    Q_ymax_override = 2e-15
    legend_outside = True

    if KE and not Ens:
        blue, lightblue, red, lightred = colors[1], colors[1], colors[1], colors[1]
    elif Ens and not KE:
        blue, lightblue, red, lightred = colors[5], colors[5], colors[5], colors[5]
   
    # set filename for saving based on True/False toggles and region name
    filename = f"figs/SWOT_dirs_comparison_{region}"
    if SF:
        filename += "_SF"
    if CG:
        filename += "_CG"
    if Ens:
        filename += "_Ens"
    if dwAw:
        filename += "_dwdAw"
    if LLL:
        filename += "_LLL"
    if Ens_LLL:
        filename += "_EnsLLL"
    if Lww:
        filename += "_Lww"
    if confint:
        filename += "_confint"
    filename += ".png"


    axes_sec = [ax.twinx() for ax in axes] if Ens and KE else None


    # Plot each dataset
    for i, (key, ds) in enumerate(datasets.items()):
        ax = axes[i]
        ax_sec = axes_sec[i] if axes_sec is not None else None
        KEflux_key = "asf_down"
        dwAw_key = "asfq_down"
        LLL_key = "LLL_down"
        Lww_key = "Lqq_down"
        x_key = "num_lines_diffs"
        dim = "swath_num"
        scale_factor = 1
        plot_dataset(ax, ax_sec, ds, dim, KEflux_key, dwAw_key, LLL_key, Lww_key, x_key, scale_factor, quantile_alpha, quantile_upper, quantile_lower, confint, SF, CG, Ens, KE, dwAw, LLL, Ens_LLL, Lww, ke_flux_label, enstrophy_flux_label, dwdAw_label, CG_label, CG_label_q, LLL_label, Ens_LLL_label, Lww_label, convert_KEFlux_to_wattperkm2permeter, (blue, lightblue, red, lightred))

    # Set common properties
    set_common_properties(axes, axes_sec, datasets, params, Ens, KE, ke_flux_label, enstrophy_flux_label, convert_KEFlux_to_wattperkm2permeter, KE_lims_scale, Q_lims_scale, draw_Rossby_radius, (blue, lightblue, red, lightred), KE_ymin_override, KE_ymax_override, Q_ymin_override, Q_ymax_override, legend_outside)

    plt.tight_layout()
    plt.savefig(filename, dpi=300)

In [ ]:
# # Plot SWOT datasets for each region. In blue plot the northeast swath direction, in red the northwest swath direction.
# for region, params in region_dict.items():

#     ds_SWOT = load_dataset("SWOT_file", params, region)
#     # Separate ds_SWOT into northeast and northwest swath directions
#     ds_SWOT_NE = ds_SWOT.where(ds_SWOT['direction'] == 'northeast', drop=True).mean('swath_num')
#     ds_SWOT_NW = ds_SWOT.where(ds_SWOT['direction'] == 'northwest', drop=True).mean('swath_num')

#     # Create subplots side by side, one plotting asf_down, one plotting asfq_down, one plotting EFlux_CG, one plotting QFlux_CG. Each with both northeast and northwest swath directions on the same plot in tab:blue and tab:red respectively.

#     fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
#     # ax1: asf_down, ax2: asfq_down, ax3: EFlux_CG, ax4: QFlux_CG

#     ax1.semilogx(2 * np.pi / ds_SWOT_NE["num_lines_diffs"],
#         -0.5 * ds_SWOT_NE["asf_down"],
#         color="tab:blue", zorder=1, label="NE swath")
#     ax1.semilogx(2 * np.pi / ds_SWOT_NW["num_lines_diffs"],
#         -0.5 * ds_SWOT_NW["asf_down"],
#         color="tab:red", zorder=1, label="NW swath")
#     ax1.set_title("KE flux from ASF")

#     ax2.semilogx(2 * np.pi / ds_SWOT_NE["num_lines_diffs"],
#         -0.5 * ds_SWOT_NE["asfq_down"],
#         color="tab:blue", zorder=1, label="NE swath")
#     ax2.semilogx(2 * np.pi / ds_SWOT_NW["num_lines_diffs"],
#         -0.5 * ds_SWOT_NW["asfq_down"],
#         color="tab:red", zorder=1, label="NW swath")
#     ax2.set_title("Enstrophy flux from dwAw")

#     ax3.semilogx(ds_SWOT_NE["K_coarse_grain"],
#         ds_SWOT_NE["EFlux_CG"],
#         color="tab:blue", zorder=1, label="NE swath")
#     ax3.semilogx(ds_SWOT_NW["K_coarse_grain"],
#         ds_SWOT_NW["EFlux_CG"],
#         color="tab:red", zorder=1, label="NW swath")
#     ax3.set_title("KE flux from Coarse Graining")

#     ax4.semilogx(ds_SWOT_NE["K_coarse_grain"],
#         ds_SWOT_NE["QFlux_CG"],
#         color="tab:blue", zorder=1, label="NE swath")
#     ax4.semilogx(ds_SWOT_NW["K_coarse_grain"],
#         ds_SWOT_NW["QFlux_CG"],
#         color="tab:red", zorder=1, label="NW swath")
#     ax4.set_title("Enstrophy flux from Coarse Graining")

### Prelim overlayed snapshots LLC4320/simSWOT/SWOT

In [ ]:
ds_LLC4320_acc_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/mitgcm/acc_data/acc_all/LLC4320_pre-SWOT_ACC_SMST_20111113.nc")
ds_simSWOT_acc_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/simulated_swot/science_phase/SWOT_L2_LR_SSH_Expert_001_008_20111113T060008_20111113T065134_DG10_01.nc")
ds_SWOT_acc_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_002/SWOT_L3_LR_SSH_Expert_002_338_20230823T031632_20230823T040758_v3.0.nc")

#cut simSWOT and SWOT based on lat/lon of LLC4320
ds_simSWOT_acc_snapshot = ds_simSWOT_acc_snapshot.where(
    (ds_simSWOT_acc_snapshot.longitude >= ds_LLC4320_acc_snapshot.XC.min()) &
    (ds_simSWOT_acc_snapshot.longitude <= ds_LLC4320_acc_snapshot.XC.max()) &
    (ds_simSWOT_acc_snapshot.latitude >= ds_LLC4320_acc_snapshot.YC.min()) &
    (ds_simSWOT_acc_snapshot.latitude <= ds_LLC4320_acc_snapshot.YC.max()),
    drop=True
)

ds_SWOT_acc_snapshot = ds_SWOT_acc_snapshot.where(
    (ds_SWOT_acc_snapshot.longitude >= ds_LLC4320_acc_snapshot.XC.min()) &
    (ds_SWOT_acc_snapshot.longitude <= ds_LLC4320_acc_snapshot.XC.max()) &
    (ds_SWOT_acc_snapshot.latitude >= ds_LLC4320_acc_snapshot.YC.min()) &
    (ds_SWOT_acc_snapshot.latitude <= ds_LLC4320_acc_snapshot.YC.max()),
    drop=True
)
# open SWOT data files and try to cut based on lat/lon of LLC4320, if error then try the next file, where all the files live at /Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_2.0/Expert/cycle_XXX/

# use glob to get all files in the directory
# files = glob.glob("/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_002/*.nc")

# i = 0
# for file in files:
#     try:
#         ds_SWOT_acc_snapshot = xr.open_dataset(file)
#         # cut based on lat/lon of LLC4320
#         ds_SWOT_acc_snapshot = ds_SWOT_acc_snapshot.where(
#             (ds_SWOT_acc_snapshot.longitude >= ds_LLC4320_acc_snapshot.XC.min()) &
#             (ds_SWOT_acc_snapshot.longitude <= ds_LLC4320_acc_snapshot.XC.max()) &
#             (ds_SWOT_acc_snapshot.latitude >= ds_LLC4320_acc_snapshot.YC.min()) &
#             (ds_SWOT_acc_snapshot.latitude <= ds_LLC4320_acc_snapshot.YC.max()),
#             drop=True
#         )

#         correlation = xr.corr(ds_SWOT_acc_snapshot["latitude"], ds_SWOT_acc_snapshot["longitude"], dim=['num_lines', 'num_pixels'])
#         if correlation < 0:
#             i += 1
#             print(i)

#             # check if lat/lon ranges of cut SWOT data are close to lat/lon ranges of simSWOT data
#             lat_diff = abs(ds_SWOT_acc_snapshot["latitude"].min() - ds_simSWOT_acc_snapshot["latitude"].min()) + abs(ds_SWOT_acc_snapshot["latitude"].max() - ds_simSWOT_acc_snapshot["latitude"].max())
#             lon_diff = abs(ds_SWOT_acc_snapshot["longitude"].min() - ds_simSWOT_acc_snapshot["longitude"].min()) + abs(ds_SWOT_acc_snapshot["longitude"].max() - ds_simSWOT_acc_snapshot["longitude"].max())
#             threshold = 1  # degrees
#             if lat_diff < threshold and lon_diff < threshold:
#                 print(f"Successfully loaded and cut SWOT data from file: {file}")
#                 break  # break if lat/lon ranges are close

#         else:
#             continue  # skip to next file if correlation is negative

#         # if i == 16:
#         #     print(f"Successfully loaded and cut SWOT data from file: {file}")
#         #     break

#         # only break if cut lat/lon is close to simSWOT lat/lon ranges
#         # lat_diff = abs(ds_SWOT_acc_snapshot["latitude"].min() - ds_simSWOT_acc_snapshot["latitude"].min()) + abs(ds_SWOT_acc_snapshot["latitude"].max() - ds_simSWOT_acc_snapshot["latitude"].max())
#         # lon_diff = abs(ds_SWOT_acc_snapshot["longitude"].min() - ds_simSWOT_acc_snapshot["longitude"].min()) + abs(ds_SWOT_acc_snapshot["longitude"].max() - ds_simSWOT_acc_snapshot["longitude"].max())
        
#         # threshold = 1  # degrees
#         # if lat_diff < threshold and lon_diff < threshold:
#         #     break  # break if cut lat/lon is close to simSWOT lat/lon ranges

#         # else:
#         #     print(f"SWOT data from file {file} does not match simSWOT lat/lon ranges after cutting. Trying next file.")
#         #     continue

#         # if successful, break the loop
#         # print(f"Successfully loaded and cut SWOT data from file: {file}")
#         # break
#     except Exception:
#         continue

In [ ]:
ds_LLC4320_acc_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/mitgcm/acc_data/acc_all/LLC4320_pre-SWOT_ACC_SMST_20111113.nc")
ds_simSWOT_acc_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/simulated_swot/science_phase/SWOT_L2_LR_SSH_Expert_001_008_20111113T060008_20111113T065134_DG10_01.nc")
ds_SWOT_acc_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_002/SWOT_L3_LR_SSH_Expert_002_338_20230823T031632_20230823T040758_v3.0.nc")

#cut simSWOT and SWOT based on lat/lon of LLC4320
ds_simSWOT_acc_snapshot = ds_simSWOT_acc_snapshot.where(
    (ds_simSWOT_acc_snapshot.longitude >= ds_LLC4320_acc_snapshot.XC.min()) &
    (ds_simSWOT_acc_snapshot.longitude <= ds_LLC4320_acc_snapshot.XC.max()) &
    (ds_simSWOT_acc_snapshot.latitude >= ds_LLC4320_acc_snapshot.YC.min()) &
    (ds_simSWOT_acc_snapshot.latitude <= ds_LLC4320_acc_snapshot.YC.max()),
    drop=True
)

ds_SWOT_acc_snapshot = ds_SWOT_acc_snapshot.where(
    (ds_SWOT_acc_snapshot.longitude >= ds_LLC4320_acc_snapshot.XC.min()) &
    (ds_SWOT_acc_snapshot.longitude <= ds_LLC4320_acc_snapshot.XC.max()) &
    (ds_SWOT_acc_snapshot.latitude >= ds_LLC4320_acc_snapshot.YC.min()) &
    (ds_SWOT_acc_snapshot.latitude <= ds_LLC4320_acc_snapshot.YC.max()),
    drop=True
)

# expand ds_SWOT_acc_snapshot along new dimension named swath_num with size 2 and duplicate the data along that dimension, so that the shape of ds_SWOT_acc_snapshot becomes (swath_num: 2, num_lines: ..., num_pixels: ...). make the latitude and longitude coordinates are also expanded
ds_SWOT_acc_snapshot = ds_SWOT_acc_snapshot.expand_dims(swath_num=[0, 1])
ds_SWOT_acc_snapshot["latitude"] = ds_SWOT_acc_snapshot["latitude"].expand_dims(swath_num=[0, 1])
ds_SWOT_acc_snapshot["longitude"] = ds_SWOT_acc_snapshot["longitude"].expand_dims(swath_num=[0, 1])

correlation = xr.corr(ds_SWOT_acc_snapshot["latitude"], ds_SWOT_acc_snapshot["longitude"], dim=["num_lines", "num_pixels",])
print(f"Correlation between latitude and longitude in ds_SWOT_acc_snapshot: {correlation.values}")
ds_SWOT_acc_snapshot["direction"] = xr.where(correlation >= 0, "northeast", "southwest")

# if direction is southwest, resort the dataset so that latitude and longitude are increasing
# direction should be 1 value per swath_num
reverse = ds_SWOT_acc_snapshot["direction"] == "southwest"

n_lines = ds_SWOT_acc_snapshot.sizes["num_lines"]
n_pixels = ds_SWOT_acc_snapshot.sizes["num_pixels"]

line_idx = xr.DataArray(np.arange(n_lines), dims="num_lines")
pix_idx  = xr.DataArray(np.arange(n_pixels), dims="num_pixels")

line_idx = xr.where(reverse, line_idx[::-1], line_idx)
pix_idx  = xr.where(reverse, pix_idx[::-1], pix_idx)

ds_SWOT_acc_snapshot = ds_SWOT_acc_snapshot.isel(
    num_lines=line_idx,
    num_pixels=pix_idx,
)

omega = 7.2921e-5
g = 9.81 
swot_dx = 2000
ds_simSWOT_acc_snapshot['f'] = 2 * omega * np.sin(ds_simSWOT_acc_snapshot.latitude * np.pi/180)
ds_SWOT_acc_snapshot['f'] = 2 * omega * np.sin(ds_SWOT_acc_snapshot.latitude * np.pi/180)

ds_simSWOT_acc_snapshot['swot_total_error'] = (
    ds_simSWOT_acc_snapshot.simulated_error_phase + 
    ds_simSWOT_acc_snapshot.simulated_error_roll + 
    ds_simSWOT_acc_snapshot.simulated_error_timing + 
    ds_simSWOT_acc_snapshot.simulated_error_baseline_dilation + 
    # ds_cut.simulated_error_karin + 
    # ds_cut.simulated_error_troposphere + 
    ds_simSWOT_acc_snapshot.simulated_error_orbital
    )

ds_simSWOT_acc_snapshot['ssh_karin_all_error_removed'] = ds_simSWOT_acc_snapshot.ssh_karin - ds_simSWOT_acc_snapshot['swot_total_error']

ds_simSWOT_acc_snapshot['ssh_karin_all_error_removed_ddnum_lines'] = ds_simSWOT_acc_snapshot['ssh_karin_all_error_removed'].differentiate('num_lines') / swot_dx
ds_simSWOT_acc_snapshot['ssh_karin_all_error_removed_ddnum_pixels'] = ds_simSWOT_acc_snapshot['ssh_karin_all_error_removed'].differentiate('num_pixels') / swot_dx
ds_simSWOT_acc_snapshot['u_geo_all_error_removed'] = - (g / ds_simSWOT_acc_snapshot['f']) * ds_simSWOT_acc_snapshot['ssh_karin_all_error_removed_ddnum_lines']
ds_simSWOT_acc_snapshot['v_geo_all_error_removed'] = (g / ds_simSWOT_acc_snapshot['f']) * ds_simSWOT_acc_snapshot['ssh_karin_all_error_removed_ddnum_pixels']

ds_simSWOT_acc_snapshot['simulated_true_ssh_karin_ddnum_lines'] = ds_simSWOT_acc_snapshot['simulated_true_ssh_karin'].differentiate('num_lines') / swot_dx
ds_simSWOT_acc_snapshot['simulated_true_ssh_karin_ddnum_pixels'] = ds_simSWOT_acc_snapshot['simulated_true_ssh_karin'].differentiate('num_pixels') / swot_dx
ds_simSWOT_acc_snapshot['u_geo'] = - (g / ds_simSWOT_acc_snapshot['f']) * ds_simSWOT_acc_snapshot['simulated_true_ssh_karin_ddnum_lines']
ds_simSWOT_acc_snapshot['v_geo'] = (g / ds_simSWOT_acc_snapshot['f']) * ds_simSWOT_acc_snapshot['simulated_true_ssh_karin_ddnum_pixels']

# Calculate geostrophic velocities and advection
ds_LLC4320_acc_snapshot["dEtadx"] = ds_LLC4320_acc_snapshot["Eta"].differentiate("i") / ds_LLC4320_acc_snapshot["DXV"]
ds_LLC4320_acc_snapshot["dEtady"] = ds_LLC4320_acc_snapshot["Eta"].differentiate("j") / ds_LLC4320_acc_snapshot["DYU"]

omega = 7.2921e-5

ds_LLC4320_acc_snapshot["f"] = 2 * omega * np.sin(ds_LLC4320_acc_snapshot.YC * np.pi / 180)

ds_LLC4320_acc_snapshot["u_geo"] = -(g / ds_LLC4320_acc_snapshot["f"]) * ds_LLC4320_acc_snapshot["dEtady"]
ds_LLC4320_acc_snapshot["v_geo"] = (g / ds_LLC4320_acc_snapshot["f"]) * ds_LLC4320_acc_snapshot["dEtadx"]

ds_SWOT_acc_snapshot["dssha_filtereddx"] = ds_SWOT_acc_snapshot["ssha_filtered"].differentiate('num_lines') / swot_dx
ds_SWOT_acc_snapshot["dssha_filtereddy"] = ds_SWOT_acc_snapshot["ssha_filtered"].differentiate('num_pixels') / swot_dx
ds_SWOT_acc_snapshot["u_geo"] = - (g / ds_SWOT_acc_snapshot['f']) * ds_SWOT_acc_snapshot['dssha_filtereddx']
ds_SWOT_acc_snapshot["v_geo"] = (g / ds_SWOT_acc_snapshot['f']) * ds_SWOT_acc_snapshot['dssha_filtereddy']

In [ ]:
plt.figure(figsize=(4,3))

plt.pcolormesh(ds_LLC4320_acc_snapshot.XC, ds_LLC4320_acc_snapshot.YC, ds_LLC4320_acc_snapshot.Eta.isel(time=0), cmap='viridis', vmin=-1, vmax=1)
plt.pcolormesh(ds_simSWOT_acc_snapshot.longitude, ds_simSWOT_acc_snapshot.latitude, ds_simSWOT_acc_snapshot.simulated_true_ssh_karin, cmap='viridis', vmin=-1, vmax=1)
plt.pcolormesh(ds_SWOT_acc_snapshot.longitude, ds_SWOT_acc_snapshot.latitude, ds_SWOT_acc_snapshot.ssha_filtered, cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='SSH (m)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.ylim(-57,-53)
# skip every other ytick label
# yticks = plt.yticks()[0]
# plt.yticks(yticks[::2])
plt.tight_layout()
# plt.savefig("figs/LLC4320_snapshots/LLC4320_simSWOT_SWOT_acc_ssh_comparison_onlyLLC4320.png", dpi=300)

In [ ]:


# ds_SWOT_acc_snapshot['u_geo'] = xr.where(ds_SWOT_acc_snapshot['direction'] == 'northeast', ds_SWOT_acc_snapshot['u_geo'], -ds_SWOT_acc_snapshot['u_geo'])
# ds_SWOT_acc_snapshot['v_geo'] = xr.where(ds_SWOT_acc_snapshot['direction'] == 'northeast', ds_SWOT_acc_snapshot['v_geo'], -ds_SWOT_acc_snapshot['v_geo'])

In [ ]:

plt.figure(figsize=(4,3))
plt.pcolormesh(ds_LLC4320_acc_snapshot.XC, ds_LLC4320_acc_snapshot.YC, ds_LLC4320_acc_snapshot.u_geo.isel(time=0), cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Geostrophic Velocity (m/s)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.ylim(-57,-53)
plt.show()

plt.figure(figsize=(4,3))
plt.pcolormesh(ds_simSWOT_acc_snapshot.longitude, ds_simSWOT_acc_snapshot.latitude, ds_simSWOT_acc_snapshot.u_geo_all_error_removed, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Geostrophic Velocity (m/s)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.ylim(-57,-53)
plt.show()

plt.figure(figsize=(4,3))
plt.pcolormesh(ds_SWOT_acc_snapshot.isel(swath_num=1).longitude, ds_SWOT_acc_snapshot.isel(swath_num=1).latitude, ds_SWOT_acc_snapshot.isel(swath_num=1).u_geo, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Geostrophic Velocity (m/s)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.ylim(-57,-53)
plt.show()

plt.figure(figsize=(4,3))
plt.pcolormesh(ds_SWOT_acc_snapshot.isel(swath_num=1).longitude, ds_SWOT_acc_snapshot.isel(swath_num=1).latitude, ds_SWOT_acc_snapshot.isel(swath_num=1).u_geo, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Geostrophic Velocity (m/s)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.ylim(-57,-53)
plt.show()

### 2026-04-08 Full simSWOT/SWOT cycle snapshot of ACC 

In [ ]:
ds_LLC4320_acc_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/mitgcm/acc_data/acc_all/LLC4320_pre-SWOT_ACC_SMST_20111113.nc")

swot_files = glob.glob("/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_001/*.nc")
simswot_files = glob.glob("/Volumes/Promise Disk/data/simulated_swot/full_data/*Expert_001*.nc")

ds_SWOT_list = []
ds_simSWOT_list = []

for files in [swot_files, simswot_files]:
    for file in files:
        try:
            ds = xr.open_dataset(file)
            # cut based on lat/lon of LLC4320
            ds = ds.where(
                (ds.longitude >= ds_LLC4320_acc_snapshot.XC.min()) &
                (ds.longitude <= ds_LLC4320_acc_snapshot.XC.max()) &
                (ds.latitude >= ds_LLC4320_acc_snapshot.YC.min()) &
                (ds.latitude <= ds_LLC4320_acc_snapshot.YC.max()),
                drop=True
            )

            if len(ds.latitude) == 0 or len(ds.longitude) == 0:
                continue

            else:
                ds_SWOT_list.append(ds) if "SWOT_L3_LR_SSH" in file else ds_simSWOT_list.append(ds)
                print(f"Successfully loaded and cut data from file: {file}")
        
        except Exception as e:
            print(f"Error loading file {file}: {e}")
            continue

In [ ]:
fig = plt.figure(figsize=(10,10))
gs = fig.add_gridspec(2, 2)

axes = [
    fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree()),  # Top row, spans both columns
    fig.add_subplot(gs[1, 0]),  # Bottom left
    fig.add_subplot(gs[1, 1]),  # Bottom right
]


# axes[0] plots a global map with stars that mark the locations of all the regions based on the latlon values in region_dict
axes[0].set_global()
axes[0].add_feature(cfeature.LAND, facecolor='0.85', zorder=0)
axes[0].coastlines(linewidth=0.8, color='0.35')
gl = axes[0].gridlines(draw_labels=True, linewidth=0.4, alpha=0.3, linestyle='--')
gl.top_labels = False
gl.right_labels = False

for region_name, params in region_dict.items():
    if 'latlon' in params and params['latlon'] is not None:
        latlon = np.asarray(params['latlon'])
        if latlon.ndim == 1 and latlon.size == 2:
            lat, lon = latlon
        elif latlon.ndim >= 2 and latlon.shape[-1] == 2:
            lat = np.nanmean(latlon[..., 0])
            lon = np.nanmean(latlon[..., 1])
        else:
            lat = 0.5 * (params['lat_north'] + params['lat_south'])
            lon = 0.5 * (params['lon_east'] + params['lon_west'])
    else:
        lat = 0.5 * (params['lat_north'] + params['lat_south'])
        lon = 0.5 * (params['lon_east'] + params['lon_west'])

    axes[0].plot(lon, lat, marker='*', color='tab:blue', markersize=12, linestyle='None', transform=ccrs.PlateCarree())

axes[1].pcolormesh(ds_LLC4320_acc_snapshot.XC, ds_LLC4320_acc_snapshot.YC, ds_LLC4320_acc_snapshot.Eta.isel(time=0), cmap='viridis')
for ds in ds_SWOT_list:
    axes[2].pcolormesh(ds.longitude, ds.latitude, ds.ssha_filtered, cmap='viridis')

axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[2].set_xlabel('Longitude')
axes[2].set_ylabel('')

# add colorbar for the bottom two subplots, with the same scale, and label it "SSH (m)", put it below the subplots stretched across the full width
cbar_ax = fig.add_axes([0.2, 0.08, 0.6, 0.02])
cbar = fig.colorbar(axes[2].collections[0], cax=cbar_ax, orientation='horizontal')
cbar.set_label('SSHA (m)')

for ax in axes[1:]:
    ax.set_ylim(-57, -53)
    ax.set_xlim(148, 158)

# add titles to each subplot
axes[0].set_title("Region locations")
axes[1].set_title("LLC4320 - ACC")
axes[2].set_title("SWOT-L3 - ACC")

plt.tight_layout()
# plt.savefig("figs/ssh_snapshots_LLC4320_SWOT_global_stars_2x2.png", dpi=300)

## 2026-05-18 Troubleshoot scaling CG vs SF

In [ ]:
ds_SWOT_snapshot = xr.open_dataset("/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_001/SWOT_L3_LR_SSH_Expert_001_149_20230726T122756_20230726T131923_v3.0.nc")

In [ ]:
ds_SWOT_snapshot

In [ ]:
ds_MITgcm_snapshot_og = xr.open_dataset("/Volumes/Promise Disk/data/mitgcm/acc_data/acc_all/LLC4320_pre-SWOT_ACC_SMST_20110913.nc")

In [ ]:
import importlib
# import module from /Users/cassswagner/OSU_research/SWOT_paper1/xarray_sf_funcs.py
import sys
sys.path.append("/Users/cassswagner/OSU_research/SWOT_paper1/scripts/")
import xarray_sf_funcs as xsfuncs

# Reload the xsfuncs module
importlib.reload(xsfuncs)

ds_MITgcm_snapshot = ds_MITgcm_snapshot_og[{"Eta", "XC", "YC", "DXV", "DYU"}]


# Calculate geostrophic velocities and advection
ds_MITgcm_snapshot["dEtadx"] = ds_MITgcm_snapshot["Eta"].differentiate("i") / ds_MITgcm_snapshot["DXV"]
ds_MITgcm_snapshot["dEtady"] = ds_MITgcm_snapshot["Eta"].differentiate("j") / ds_MITgcm_snapshot["DYU"]

omega = 7.2921e-5
g = 9.81

ds_MITgcm_snapshot["f"] = 2 * omega * np.sin(ds_MITgcm_snapshot.YC * np.pi / 180)

ds_MITgcm_snapshot["u_geo"] = -(g / ds_MITgcm_snapshot["f"]) * ds_MITgcm_snapshot["dEtady"]
ds_MITgcm_snapshot["v_geo"] = (g / ds_MITgcm_snapshot["f"]) * ds_MITgcm_snapshot["dEtadx"]

print("Computed geostrophic velocities")

ecco_dx_est = ds_MITgcm_snapshot["DXV"].mean().values
ecco_dy_est = ds_MITgcm_snapshot["DYU"].mean().values

ds_MITgcm_snapshot = xsfuncs.calc_advection(
    ds_MITgcm_snapshot,
    qvar="CALC",
    uvar="u_geo",
    vvar="v_geo",
    xvar="i",
    yvar="j",
    dx=ds_MITgcm_snapshot["DXV"],
    dy=ds_MITgcm_snapshot["DYU"],
)

ds_MITgcm_snapshot_sfs = xsfuncs.compute_asf(
                ds_MITgcm_snapshot,
                shift_func="shift",
                uvar="u_geo",
                vvar="v_geo",
                qvar=None,
                xvar="i",
                yvar="j",
                shiftnum=1,
                full=True,
                just_mean=True,
            )

In [ ]:
ds_MITgcm_snapshot_shiftby1 = xsfuncs.shift_fields(ds_MITgcm_snapshot, var='u_geo', shifts = {"i_left": {"i": -1} , "j_down": {"j": -1}}, shift_func="shift")

In [ ]:
ds_MITgcm_snapshot_shiftby1['u_geo_shifted_j_down'].isel(i=slice(51,54), j=slice(51,54), time=0)

In [ ]:
ds_MITgcm_snapshot.isel(i=slice(51,54), j=slice(51,54), time=0).u_geo.compute()

In [ ]:
### MITGCM ###############
ds_MITgcm_acc_noprocessing = xr.open_dataset(
                                        "../data/MITgcm/2026-05-18/acc/20260518_161437_LLC4320_acc_20110913_LLL_Bessels_tapered_CG.nc")
ds_MITgcm_acc_coarsen = xr.open_dataset(
                                        "../data/MITgcm/2026-05-18/acc/20260518_155509_LLC4320_acc_20110913_coarsen_to_1div9.6deg_LLL_Bessels_tapered_CG.nc")
ds_MITgcm_acc_meanremoval = xr.open_dataset(
                                        "../data/MITgcm/2026-05-18/acc/20260518_163316_LLC4320_acc_20110913_timemean_removed_snapshot_mean_LLL_Bessels_tapered_CG.nc")
ds_MITgcm_acc_coarsen_meanremoval = xr.open_dataset(
                                        "../data/MITgcm/2026-05-18/acc/20260518_162517_LLC4320_acc_20110913_coarsen_to_1div9.6deg_timemean_removed_snapshot_mean_LLL_Bessels_tapered_CG.nc")

### simSWOT ###############
ds_simSWOT_acc_noprocessing = xr.open_dataset(
                                        "../data/simSWOT/science_phase/2026-05-18/acc/20260518_162524_simSWOT_acc_cycle_001_LLL_Bessels_tapered_CG.nc")
ds_simSWOT_acc_coarsen = xr.open_dataset(
                                        "../data/simSWOT/science_phase/2026-05-18/acc/20260518_161656_simSWOT_acc_cycle_001_coarsen_to_1div9.6deg_q_Lqq_Bessels_tapered_CG.nc")
ds_simSWOT_acc_meanremoval = xr.open_dataset(
                                        "../data/simSWOT/science_phase/2026-05-18/acc/20260518_164440_simSWOT_acc_cycle_001_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG.nc")
ds_simSWOT_acc_science_coarsen_meanremoval = xr.open_dataset(
                                        "../data/simSWOT/science_phase/2026-05-18/acc/20260518_165017_simSWOT_acc_cycle_001_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG.nc")

### SWOT ###############
ds_SWOT_acc_noprocessing = xr.open_dataset(
                                        "../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-18/science_phase/acc/20260518_161016_SWOT_L3_acc_cycle_001_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels.nc")
ds_SWOT_acc_coarsen = xr.open_dataset(
                                        "../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-18/science_phase/acc/20260518_155355_SWOT_L3_acc_cycle_001_coarsen_to_1div9.6deg_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels.nc")
ds_SWOT_acc_meanremoval = xr.open_dataset(
                                        "../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-18/science_phase/acc/20260518_161901_SWOT_L3_acc_cycle_001_timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels.nc")
ds_SWOT_acc_science_coarsen_meanremoval = xr.open_dataset(
                                        "../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-18/science_phase/acc/20260518_162113_SWOT_L3_acc_cycle_001_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels.nc")


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 15), sharex=True, sharey=True)

datasets = {
    "MITgcm": {
        "original": ds_MITgcm_acc_noprocessing,
        "coarsen to 10km": ds_MITgcm_acc_coarsen,
        "mean removed": ds_MITgcm_acc_meanremoval,
        "coarsen and mean removed": ds_MITgcm_acc_coarsen_meanremoval,
    },
    "simSWOT": {
        "original": ds_simSWOT_acc_noprocessing,
        "coarsen to 10km": ds_simSWOT_acc_coarsen,
        "mean removed": ds_simSWOT_acc_meanremoval,
        "coarsen and mean removed": ds_simSWOT_acc_science_coarsen_meanremoval,
    },
    "SWOT": {
        "original": ds_SWOT_acc_noprocessing,
        "coarsen to 10km": ds_SWOT_acc_coarsen,
        "mean removed": ds_SWOT_acc_meanremoval,
        "coarsen and mean removed": ds_SWOT_acc_science_coarsen_meanremoval,
    },
}

scale_factor = 1 # density of seawater in kg/m^3, convert from m^2/s^3 to W/m^3 by multiplying by density
for i, (dataset_name, dataset_versions) in enumerate(datasets.items()):
    for j, (version_name, ds) in enumerate(dataset_versions.items()):
        dim = "swath_num" if "swath_num" in ds.dims else "time"
        x_key = "num_lines_diffs" if "num_lines_diffs" in ds else "i_diffs"
        ax = axes[i, j]
        ax.semilogx(ds["K_coarse_grain"] / np.pi, ds["EFlux_CG"].mean(dim=dim) * scale_factor, label="CG", color="k", lw=2)
        ax.semilogx(1 / ds[x_key], -0.5 * ds["asf_mean"].mean(dim=dim) * scale_factor, label="ASF", color="tab:blue")
        ax.semilogx(1 / ds[x_key], -(2/(3 * ds[x_key])) * ds["LLL_mean"].mean(dim=dim) * scale_factor, label="LLL", color="tab:red")
        ax.set_title(f"{dataset_name} - {version_name}")
        ax.hlines(0, 1e-6, 1e-3, colors='k', lw=1)
        ax.set_xlim(1e-6, 1e-3)
        ax.set_xlabel("Wavenumber (1/m)") if i == 2 else None
        y_units = r"W/m$^3$" if scale_factor != 1 else r"m$^2$/s$^3$"
        ax.set_ylabel(f"KE Flux ({y_units})") if j == 0 else None
        ax.set_ylim(-9e-6, 3e-6) if scale_factor != 1 else ax.set_ylim(-1e-7 * scale_factor, 1e-7 * scale_factor)
        # use scientific notation for y-axis labels
        ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

        # draw wang hlines if scale_factor is not 1
        if scale_factor != 1:
            ax.hlines(-7e-6, 1e-6, 1e-3, colors='k', lw=1, linestyles='--', label='Wang et al. (2025)') if dataset_name == "MITgcm" else ax.hlines(-3.5e-6, 1e-6, 1e-3, colors='k', lw=1, linestyles='--', label='Wang et al. (2025)')
        ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig("../figures/2026-05-18_LLC-simSWOT-SWOT_science_processing_comparison_Wm3.png", dpi=300) if scale_factor != 1 else plt.savefig("../figures/2026-05-18_LLC-simSWOT-SWOT_science_processing_comparison_m2s3.png", dpi=300)

### Coarse and timemean comparison multiple regions

In [ ]:
# print min amd max of ds.time where ds.time also include NaT
for dataset_name, dataset_versions in datasets.items():
    for version_name, ds in dataset_versions.items():
        # print(ds.dims)
        if "time" in ds.dims or "time" in ds:
            time_values = ds["time"].values
            time_values = time_values[~pd.isnull(time_values)]
            if len(time_values) > 0:
                min_time = np.min(time_values)
                max_time = np.max(time_values)

                if min_time == 0:
                    ds = ds.assign_coords(time=pd.date_range(start="2011-09-13", periods=ds.sizes['time'], freq='h'))
                    min_time = ds.time.min().values
                    max_time = ds.time.max().values

                # print the min and max time values in a human readable format only date
                print(f"{dataset_name} - {version_name}: Min time = {pd.to_datetime(min_time).date()}, Max time = {pd.to_datetime(max_time).date()}")   
            else:
                print(f"{dataset_name} - {version_name}: No valid time values found.")
        else:
            print(f"{dataset_name} - {version_name}: No 'time' dimension found.")

In [ ]:
from pathlib import Path

regions = {
    "acc": {
        "full_name": "Antarctic Circumpolar Current",
        "asf_zoom": 0.75,
        "cg_zoom": 1e1,
    },
    "capebasin": {
        "full_name": "Cape Basin",
        "asf_zoom": 1,
        "cg_zoom": 1e1,
    },
    "labradorsea": {
        "full_name": "Labrador Sea",
        "asf_zoom": 0.6,
        "cg_zoom": 1e1,
    },
    "newcaledonia": {
        "full_name": "New Caledonia",
        "asf_zoom": 0.5,
        "cg_zoom": 1e1,
    },
    "nwaustralia": {
        "full_name": "Northwest Australia",
        "asf_zoom": 1,
        "cg_zoom": 1e1,
    },
    "nwpacific": {
        "full_name": "Northwest Pacific",
        "asf_zoom": 1,
        "cg_zoom": 1e1,
    },
    "westatlantic": {
        "full_name": "West Atlantic",
        "asf_zoom": 1,
        "cg_zoom": 1e1,
    },
}

dataset_config = {
    "MITgcm": {
        "file_prefix": "LLC4320",
        "base_dir": Path("../data/MITgcm/2026-05-27"),
        "mid_str": "merged_dates_20110913-20110919",
        "coarsen_suffixes": {
            "timemean removed": "timemean_removed_snapshot_mean_LLL_Bessels_tapered_CG",
            "coarsened + timemean removed": "coarsen_to_1div9.6deg_timemean_removed_snapshot_mean_LLL_Bessels_tapered_CG",
        },
    },
    "SWOT": {
        "file_prefix": "SWOT_L3",
        "base_dir": Path("../data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-27/science_phase"),
        "mid_str": "cycles_001-005",
        "coarsen_suffixes": {
            "timemean removed": "timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels",
            "coarsened + timemean removed": "coarsen_to_1div9.6deg_timemean_removed_cycle_mean_flipped_swath_LLL_Bessels_tapered_CG_filtered_vels",
        },
    },
    "simSWOT": {
        "file_prefix": "simSWOT",
        "base_dir": Path("../data/simSWOT/2026-05-27/science_phase"),
        "mid_str": "cycles_001-005",
        "coarsen_suffixes": {
            "timemean removed": "timemean_removed_cycle_mean_LLL_Bessels_tapered_CG",
            "coarsened + timemean removed": "coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG",
        },
    },
}

scale_to_cg = False

def find_single_match(base_dir, pattern):
    matches = sorted(base_dir.rglob(pattern))
    if len(matches) != 1:
        raise ValueError(f"Expected one match for {pattern} under {base_dir}, found {len(matches)}")
    return str(matches[0])


for region, region_data in regions.items():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True)

    dataset_paths = {
        dataset_name: {
            version_name: f"{config['file_prefix']}_{region}_{config['mid_str']}_{suffix}.nc"
            for version_name, suffix in config["coarsen_suffixes"].items()
        }
        for dataset_name, config in dataset_config.items()
    }

    datasets = {
        dataset_name: {
            version_name: xr.open_dataset(
                find_single_match(dataset_config[dataset_name]["base_dir"] / region, file_pattern)
            )
            for version_name, file_pattern in version_patterns.items()
        }
        for dataset_name, version_patterns in dataset_paths.items()
    }

    scale_factor = 1
    row_versions = ["timemean removed", "coarsened + timemean removed"]
    column_order = ["MITgcm", "simSWOT", "SWOT"]

    for i, version_name in enumerate(row_versions):
        for j, dataset_name in enumerate(column_order):
            ds = datasets[dataset_name][version_name]
            dim = "swath_num" if "swath_num" in ds.dims else "time"
            x_key = "num_lines_diffs" if "num_lines_diffs" in ds else "i_diffs"
            ax = axes[i, j]
            ax.semilogx(ds["L"] * 2e3 / np.pi , ds["EFlux_CG"].mean(dim=dim) * scale_factor, label="CG", color="k", lw=2)
            ax.semilogx(ds[x_key], -0.5 * ds["asf_mean"].mean(dim=dim) * scale_factor, label="ASF", color="tab:blue")
            ax.semilogx(ds[x_key], -(2 / (3 * ds[x_key])) * ds["LLL_mean"].mean(dim=dim) * scale_factor, label="LLL", color="tab:red")
            # compute date range for the dataset (use logic similar to the inspection cell)
            try:
                if "time" in ds.dims or "time" in ds:
                    time_values = ds["time"].values
                    time_values = time_values[~pd.isnull(time_values)]
                    if len(time_values) > 0:
                        if np.min(time_values) == 0:
                            ds = ds.assign_coords(time=pd.date_range(start="2011-09-13", periods=ds.sizes['time'], freq='h'))
                            time_values = ds["time"].values
                            time_values = time_values[~pd.isnull(time_values)]
                        min_date = pd.to_datetime(time_values.min()).date()
                        max_date = pd.to_datetime(time_values.max()).date()
                        date_range = f"{min_date} to {max_date}"
                    else:
                        date_range = "no time"
                else:
                    date_range = "no time"
            except Exception:
                date_range = "unknown"
            ax.set_title(f"{dataset_name} - {version_name} \n {date_range}")
            # ax.hlines(0, 1e-6, 1e-3, colors="k", lw=1)
            # ax.set_xlim(1e-6, 1e-3)
            # ax.set_xlabel("Wavenumber (1/m)") if i == 1 else None
            y_units = r"W/m$^3$" if scale_factor != 1 else r"m$^2$/s$^3$"
            ax.set_ylabel(f"KE Flux ({y_units})") if j == 0 else None
            # set ymin ymax based on the SWOT ylimits
            # SWOT_ymin = (datasets["SWOT"]["timemean removed"]["EFlux_CG"].mean(dim="swath_num")).min().values
            # SWOT_ymax = (datasets["SWOT"]["timemean removed"]["EFlux_CG"].mean(dim="swath_num")).max().values
            # SWOT_ymin = (-0.5 * datasets["SWOT"]["timemean removed"]["asf_mean"].mean(dim="swath_num")).min().values
            # SWOT_ymax = (-0.5 * datasets["SWOT"]["timemean removed"]["asf_mean"].mean(dim="swath_num")).max().values
            # ax.set_ylim(SWOT_ymin * 1.5, SWOT_ymax * 1.5)
            ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

            if scale_factor != 1:
                ax.hlines(-7e-6, 1e-6, 1e-3, colors="k", lw=1, linestyles="--", label="Wang et al. (2025)") if dataset_name == "MITgcm" else ax.hlines(-3.5e-6, 1e-6, 1e-3, colors="k", lw=1, linestyles="--", label="Wang et al. (2025)")
            ax.legend(loc="upper right")

    fig.suptitle(region_data["full_name"], fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    # plt.savefig(f"../figures/2026-05-29_allregions_alldatasets_allmethods/2026-05-29_LLC-simSWOT-SWOT_science_processing_comparison_{region}.png", dpi=300)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

for i, ds in enumerate([ds_MITgcm, ds_simSWOT, ds_SWOT]):
    ds_name = "MITgcm" if ds is ds_MITgcm else ("simSWOT" if ds is ds_simSWOT else "SWOT")
    dim = "swath_num" if "swath_num" in ds.dims else "time"
    x_key = "num_lines_diffs" if "num_lines_diffs" in ds else "i_diffs"
    abval = False
    sf_dir = "mean"
    ax = axes[i]
    ax.plot(
        ds["K_coarse_grain"] / np.pi,
        np.abs(ds["EFlux_CG"].mean(dim=dim)) if abval else ds["EFlux_CG"].mean(dim=dim),
        label="CG",
        color="tab:blue",
    )
    ax.plot(
        1 / ds[x_key],
        np.abs(-0.5 * ds[f"asf_{sf_dir}"].mean(dim=dim)) if abval else -0.5 * ds[f"asf_{sf_dir}"].mean(dim=dim),
        label="ASF",
        color="tab:red"
        )
    ax.plot(
        1 / ds[x_key],
        np.abs(-(2/(3 * ds[x_key])) * ds[f"LLL_{sf_dir}"].mean(dim=dim)) if abval else -(2/(3 * ds[x_key])) * ds[f"LLL_{sf_dir}"].mean(dim=dim),
        label="LLL",
        color="tab:green"
        )

    # if abval is True, plot -flux as 'x'
    if abval:
        ax.plot(
            ds["K_coarse_grain"] / np.pi,
            -ds["EFlux_CG"].mean(dim=dim),
            marker='x', linestyle='None', color="tab:blue"
        )
        ax.plot(
            1 / ds[x_key],
            0.5 * ds["asf_down"].mean(dim=dim),
            marker='x', linestyle='None', color="tab:red"
        )
        ax.plot(
            1 / ds[x_key],
            (2/(3 * ds[x_key]) * ds["LLL_down"].mean(dim=dim)),
            marker='x', linestyle='None', color="tab:green"
        )

    ax.set_xscale("log")
    ax.set_yscale("log") if abval else None
    ax.set_ylabel(r"KE flux (m$^2$/s$^3$)")
    ax.set_xlabel("Wavenumber (1/m)")
    ax.legend()
    ax.set_title(ds_name)
    ax.hlines(0, 1e-6, 1e-3, colors='k', lw=1)
    ax.set_xlim(1e-6, 1e-3)
    ax.set_ylim(-1e-7, 1e-7)
plt.show()

## Test sandbox

### 2026-04-08 Test linear fit large-scale mean removal

In [ ]:
ds_simSWOT = xr.open_dataset("/Volumes/Promise Disk/data/simulated_swot/full_data/SWOT_L2_LR_SSH_Expert_001_008_20111113T060008_20111113T065134_DG10_01.nc")

ds_simSWOT["simulated_true_ssh_karin_numpixels_mean"] = ds_simSWOT["simulated_true_ssh_karin"].mean(dim=["num_pixels"], skipna=True)

# Fit a line to the mean values along xvar using xarray DataArray.polyfit
coeffs = ds_simSWOT["simulated_true_ssh_karin_numpixels_mean"].polyfit(dim="num_lines", deg=1, skipna=True)
# Extract the slope and intercept from the coefficients
slope = coeffs.sel(degree=1)
intercept = coeffs.sel(degree=0)

# Calculate the fitted line values
fitted_line = slope * ds_simSWOT["num_lines"] + intercept

# Expand fitted line to have dimensions (num_pixels, num_lines)
fitted_line = fitted_line.expand_dims(num_pixels=ds_simSWOT["num_pixels"], axis=1)
fitted_line = fitted_line.transpose("num_lines", "num_pixels")

# assign fitted_line as a variable in ds_simSWOT and it should have the same latlon coordinates/dims as simulated_true_ssh_karin
ds_simSWOT["fitted_line"] = (("num_lines", "num_pixels"), fitted_line.polyfit_coefficients.data)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

vmin, vmax = -2, 0.5

axes[0].pcolormesh(ds_simSWOT.longitude, ds_simSWOT.latitude, ds_simSWOT.simulated_true_ssh_karin, cmap='viridis', vmin=vmin, vmax=vmax)
axes[1].pcolormesh(ds_simSWOT.longitude, ds_simSWOT.latitude, ds_simSWOT.fitted_line, cmap='viridis', vmin=vmin, vmax=vmax)
axes[2].pcolormesh(ds_simSWOT.longitude, ds_simSWOT.latitude, ds_simSWOT.simulated_true_ssh_karin - ds_simSWOT.fitted_line, cmap='viridis', vmin=vmin, vmax=vmax)
plt.ylim(-57, -53)
plt.xlim(148, 158)

# add colorbar
cbar0 = fig.colorbar(axes[0].collections[0], ax=axes[0], orientation='vertical', pad=0.05)
cbar0.set_label('SSH (m)')

cbar1 = fig.colorbar(axes[1].collections[0], ax=axes[1], orientation='vertical', pad=0.05)
cbar1.set_label('SSH large scale mean (m)')

cbar2 = fig.colorbar(axes[2].collections[0], ax=axes[2], orientation='vertical', pad=0.05)
cbar2.set_label('SSH - large scale mean (m)')

In [ ]:
ds_simSWOT_regular_timemean_removed = xr.open_dataset("data/simSWOT/2026-04-07/acc/20260407_164049_simSWOT_acc_cycle_001_timemean_removed_Bessels_tapered_CG.nc")
ds_simSWOT_linearfit_timemean_removed = xr.open_dataset("data/simSWOT/2026-04-07/acc/20260408_165111_simSWOT_acc_cycle_001_timemean_removed_linear_fit_Bessels_tapered_CG.nc")
ds_simSWOT_no_timemean_removed = xr.open_dataset("data/simSWOT/2026-04-07/acc/20260407_161941_simSWOT_acc_cycle_001_Bessels_tapered_CG.nc")
ds_simSWOT_cyclemean_removed = xr.open_dataset("data/simSWOT/2026-04-07/acc/20260412_135634_simSWOT_acc_cycle_001_timemean_removed_cycle_mean_Bessels_tapered_CG.nc")

In [ ]:
qmin = 0.01
qmax = 0.99
alpha = 0.5

# ASF parameters
yvar="asf_left"
xvar="num_lines_diffs"
scale_factor = -0.5
xlim_scale = 1e2

# # CG parameters
# yvar="EFlux_CG"
# xvar="K_coarse_grain"
# scale_factor = 1
# xlim_scale = 1e3

# # Bessel parameters
# yvar = "EFlux_Bessel_ASF_mean_tapered"
# xvar = "K"
# scale_factor = 1
# xlim_scale = 3e2

for ds in [ds_simSWOT_no_timemean_removed, ds_simSWOT_regular_timemean_removed, ds_simSWOT_linearfit_timemean_removed, ds_simSWOT_cyclemean_removed]:
    ds["EFlux_Bessel_ASF_mean_tapered"] = (sum(ds[var] for var in ds.data_vars if "Bessel_asf" in var)/4)
    ds["EFlux_Bessel_ASF_mean_tapered"] = (sum(ds[var] for var in ds.data_vars if "Bessel_asf" in var)/4)
    ds["EFlux_Bessel_ASF_mean_tapered"] = (sum(ds[var] for var in ds.data_vars if "Bessel_asf" in var)/4)

plt.semilogx(
    ds_simSWOT_no_timemean_removed[xvar],
    scale_factor * ds_simSWOT_no_timemean_removed[yvar].mean("swath_num"),
    color="tab:blue",
    label="Original velocities",
)

plt.fill_between(
    ds_simSWOT_no_timemean_removed[xvar],
    scale_factor * ds_simSWOT_no_timemean_removed[yvar].quantile(qmin, dim="swath_num"),
    scale_factor * ds_simSWOT_no_timemean_removed[yvar].quantile(qmax, dim="swath_num"),
    facecolor="none",
    edgecolor="tab:blue",
    hatch="o",
    linewidth=0.8,
    alpha=alpha,
)

#=======#
plt.semilogx(
    ds_simSWOT_regular_timemean_removed[xvar],
    scale_factor * ds_simSWOT_regular_timemean_removed[yvar].mean("swath_num"),
    color="tab:red",
    label="Multi-swath mean removed",
)

plt.fill_between(
    ds_simSWOT_regular_timemean_removed[xvar],
    scale_factor * ds_simSWOT_regular_timemean_removed[yvar].quantile(qmin, dim="swath_num"),
    scale_factor * ds_simSWOT_regular_timemean_removed[yvar].quantile(qmax, dim="swath_num"),
    facecolor="none",
    edgecolor="tab:red",
    hatch="\\\\",
    linewidth=0.8,
    alpha=alpha,
)

#=======#
plt.semilogx(
    ds_simSWOT_linearfit_timemean_removed[xvar],
    scale_factor * ds_simSWOT_linearfit_timemean_removed[yvar].mean("swath_num"),
    color="tab:green",
    label="Large-scale mean flow removed",
)

plt.fill_between(
    ds_simSWOT_linearfit_timemean_removed[xvar],
    scale_factor * ds_simSWOT_linearfit_timemean_removed[yvar].quantile(qmin, dim="swath_num"),
    scale_factor * ds_simSWOT_linearfit_timemean_removed[yvar].quantile(qmax, dim="swath_num"),
    facecolor="none",
    edgecolor="tab:green",
    hatch="///",
    linewidth=0.8,
    alpha=alpha,
)

#======#
plt.semilogx(
    ds_simSWOT_cyclemean_removed[xvar],
    scale_factor * ds_simSWOT_cyclemean_removed[yvar].mean("swath_num"),
    color="tab:orange",
    label="Cycle mean removed",
)

plt.fill_between(
    ds_simSWOT_cyclemean_removed[xvar],
    scale_factor * ds_simSWOT_cyclemean_removed[yvar].quantile(qmin, dim="swath_num"),
    scale_factor * ds_simSWOT_cyclemean_removed[yvar].quantile(qmax, dim="swath_num"),
    facecolor="none",
    edgecolor="tab:orange",
    hatch="xxx",
    linewidth=0.8,
    alpha=alpha,
)

plt.hlines(0, ds_simSWOT_regular_timemean_removed[xvar][1:].min(), xlim_scale * ds_simSWOT_regular_timemean_removed[xvar][1:].min(), colors="k", lw=1, zorder=0)

if yvar == "asf_mean":
    plt.xlabel("Separation distance (m)")
    plt.ylabel("KE flux from ASF (m$^2$/s$^3$)")
elif yvar == "EFlux_CG":
    plt.xlabel("Wavenumber (m$^{-1}$)")
    plt.ylabel("KE flux from coarse graining (m$^2$/s$^3$)")
plt.legend(loc="upper left")
plt.xlim(ds_simSWOT_regular_timemean_removed[xvar][1:].min(), xlim_scale * ds_simSWOT_regular_timemean_removed[xvar][1:].min());
plt.ylim(-4e-6, 4e-6);

### 2026-04-11 Test simSWOT common swath mean

In [ ]:
import glob
import os
import re
from collections import defaultdict

base_dir = "/Volumes/Promise Disk/data/simulated_swot/fast_phase"
target_years = {"2011", "2012"}
run_new = False

file_re = re.compile(r"SWOT_L2_LR_SSH_Expert.*_(\d{3})_(\d{4}).*\.nc$")
files_by_pass = defaultdict(list)

# One scan across the directory; bucket files by pass for target years only.
for file_path in glob.iglob(f"{base_dir}/SWOT_L2_LR_SSH_Expert*.nc"):
    m = file_re.search(os.path.basename(file_path))
    if m is None:
        continue

    pass_tag, year = m.group(1), m.group(2)
    if year in target_years:
        files_by_pass[pass_tag].append(file_path)

if not files_by_pass:
    raise FileNotFoundError(f"No simSWOT files found in {base_dir} for years {sorted(target_years)}")

n_passes = len(files_by_pass)
n_files = sum(len(v) for v in files_by_pass.values())
print(f"Found {n_files} files across {n_passes} passes for {sorted(target_years)}")

for pass_tag in sorted(files_by_pass):
    mean_file = f"{base_dir}/SWOT_L2_LR_SSH_Expert_Pass-{pass_tag}_Mean_2011-2012.nc"
    if os.path.exists(mean_file):
        if not run_new:
            print(f"Mean file for pass {pass_tag} already exists at {mean_file}, skipping.")
            continue
        else:
            # Delete existing mean file to replace with new calculation
            print(f"Mean file for pass {pass_tag} already exists at {mean_file}, but run_new is True, so it will be overwritten.")
            os.remove(mean_file)

    file_list = sorted(files_by_pass[pass_tag])
    print(f"Pass {pass_tag}: opening {len(file_list)} files")

    ds = xr.open_mfdataset(
        file_list,
        combine="nested",
        concat_dim="cycle",
    )

    print(f"Pass {pass_tag}: calculating mean")
    ds_mean = ds.mean(dim="cycle", keep_attrs=True, skipna=True)

    # print(f"Pass {pass_tag}: adding pass_number attribute and writing to {mean_file}")
    ds_mean.attrs["pass_number"] = pass_tag
    print(f"Pass {pass_tag}: writing {mean_file}")
    ds_mean.to_netcdf(mean_file)

    ds.close()
    ds_mean.close()

### 2026-05-04 creating SWOT means

In [ ]:
import glob
import os
import re
from collections import defaultdict

base_dir = "/Volumes/Promise Disk/data/validated_swot"
cycle_range = range(1, 49)  # Passes 001-048 inclusive
target_cycles = set(str(i).zfill(3) for i in cycle_range)  # Passes 001-048 inclusive
run_new = False

file_re = re.compile(r"SWOT_L3_LR_SSH_Expert_(\d{3})_(\d{3}).*\.nc$")
files_by_pass = defaultdict(list)

glob_list = glob.glob("/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/*/*/*.nc")

# One scan across the directory; bucket files by pass for target cycles only.
for file in glob_list:
    m = file_re.search(os.path.basename(file))
    if m is None:
        continue
    
    # print(m)
    # print(m.group(0), m.group(1), m.group(2))
    # break
    cycle, pass_tag = m.group(1), m.group(2)
    # print(f"Found file for pass {pass_tag}, cycle {cycle}: {file}")
    if cycle in target_cycles:
        files_by_pass[pass_tag].append(file)

if not files_by_pass:
    raise FileNotFoundError(f"No SWOT files found in {base_dir} for cycles {sorted(target_cycles)}")

n_passes = len(files_by_pass)
n_files = sum(len(v) for v in files_by_pass.values())
# print(f"Found {n_files} files across {n_passes} passes for {sorted(target_cycles)}")

# Only process passes in range 

for pass_tag in sorted(files_by_pass):
    mean_file = f"{base_dir}/SWOT_L3_LR_SSH_Expert_Pass-{pass_tag}_Mean_cycles-{cycle_range.start:03d}-{cycle_range.stop-1:03d}_v3.0.nc"
    if os.path.exists(mean_file):
        if not run_new:
            print(f"Mean file for pass {pass_tag} already exists at {mean_file}, skipping.")
            continue
        else:
            # Delete existing mean file to replace with new calculation
            print(f"Mean file for pass {pass_tag} already exists at {mean_file}, but run_new is True, so it will be overwritten.")
            os.remove(mean_file)

    file_list = sorted(files_by_pass[pass_tag])

    # drop files that are outside of the cycle range 001-048
    file_list = [f for f in file_list if any(cycle in f for cycle in target_cycles)]

    print(f"Pass {pass_tag}: opening {len(file_list)} files")

    ds = xr.open_mfdataset(
        file_list,
        combine="nested",
        drop_variables=["i_num_line", "i_num_pixel"],
        concat_dim="cycle",
    )

    print(f"Pass {pass_tag}: calculating mean")
    ds_mean = ds.mean(dim="cycle", keep_attrs=True, skipna=True)

    # print(f"Pass {pass_tag}: adding pass_number attribute and writing to {mean_file}")
    ds_mean.attrs["pass_number"] = pass_tag
    print(f"Pass {pass_tag}: writing {mean_file}")
    ds_mean.to_netcdf(mean_file)

    ds.close()
    ds_mean.close()

In [ ]:
ds02 = xr.open_dataset(file_list[1])

In [ ]:
ds01.drop_vars(["i_num_line", "i_num_pixel"])

In [ ]:
ds02

In [ ]:
base_dir = "/Volumes/Promise Disk/data/validated_swot/fast_phase"
target_years = {"2023", "2024", "2025", "2026"}
run_new = False

file_re = re.compile(r"SWOT_L3_LR_SSH_Expert.*_(\d{3})_(\d{4}).*\.nc$")
files_by_pass = defaultdict(list)

In [ ]:
file_re

In [ ]:
ds_simSWOT = xr.open_dataset("/Volumes/Promise Disk/data/simulated_swot/full_data/SWOT_L2_LR_SSH_Expert_001_008_20111113T060008_20111113T065134_DG10_01.nc")
ds_simSWOT_cycle001 = xr.open_mfdataset("/Volumes/Promise Disk/data/simulated_swot/full_data/*Expert_001_00*.nc", combine="nested", concat_dim="pass_number")

In [ ]:
ds_simSWOT_means_pass001_008 = xr.open_mfdataset(["/Volumes/Promise Disk/data/simulated_swot/full_data/SWOT_L2_LR_SSH_Expert_Pass-001_Mean_2011-2012.nc", "/Volumes/Promise Disk/data/simulated_swot/full_data/SWOT_L2_LR_SSH_Expert_Pass-008_Mean_2011-2012.nc"], combine="nested", concat_dim="pass_number", preprocess=lambda ds: ds.expand_dims(pass_number=[int(ds.attrs["pass_number"])]), combine_attrs="drop_conflicts").sortby("pass_number")

In [ ]:
ds_simSWOT_cycle001

In [ ]:
ds_simSWOT_means_pass001_008

In [ ]:
for pass_num in ds_simSWOT_cycle001_meanremoved.pass_number.values:
   # plot all 3 datasets for this pass_num with x,y as lonlat and color as ssh, and make sure all 3 subplots have the same x and y limits and colorbar limits
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
    ds_simSWOT_cycle001.simulated_true_ssh_karin.sel(pass_number=pass_num).plot.pcolormesh(x='longitude', y='latitude', cmap='viridis', vmin=-2, vmax=0.5, ax=axes[0], add_colorbar=False)
    ds_simSWOT_means_pass001_008.simulated_true_ssh_karin.sel(pass_number=pass_num).plot.pcolormesh(x='longitude', y='latitude', cmap='viridis', vmin=-2, vmax=0.5, ax=axes[1], add_colorbar=False)
    ds_simSWOT_cycle001_meanremoved.simulated_true_ssh_karin.sel(pass_number=pass_num).plot.pcolormesh(x='longitude', y='latitude', cmap='viridis', vmin=-2, vmax=0.5, ax=axes[2], add_colorbar=False)
    # plt.ylim(-57, -53)
    # plt.xlim(148, 158)
    # add colorbar to the right of the last subplot only
    cbar = fig.colorbar(axes[2].collections[0], ax=axes[2], orientation='vertical', pad=0.05)
    cbar.set_label('SSH (m)')
    # add titles to each subplot
    axes[0].set_title(f"simSWOT Pass {pass_num} - Original")
    axes[1].set_title(f"simSWOT Pass {pass_num} - Multi-pass mean")
    axes[2].set_title(f"simSWOT Pass {pass_num} - Original - Multi-pass mean")
    plt.tight_layout()
    plt.show()


In [ ]:
ds_simSWOT_cycle001.isel(pass_number=1).simulated_true_ssh_karin.plot(x="longitude", y="latitude", cmap="viridis")
ds_simSWOT_cycle001.isel(pass_number=0).simulated_true_ssh_karin.plot(x="longitude", y="latitude", cmap="viridis")

### 2026-05-06 get pass_number

In [ ]:
path = "/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_001/SWOT_L3_LR_SSH_Expert_001_149_20230726T122756_20230726T131923_v3.0.nc"

ds_SWOT = xr.open_dataset(path)

In [ ]:
# extract pass number from filename, with example filename SWOT_L3_LR_SSH_Expert_001_149_20230726T122756_20230726T131923_v3.0.nc where 149 is the pass number
filename = path.split("/")[-1]
pass_value = int(filename.split("_")[6])
print(f"Extracted pass number: {pass_value}")

### Random testing

In [ ]:
ds_tmp = xr.open_dataset("/Volumes/Promise Disk/data/mitgcm/acc_data/acc_all/LLC4320_pre-SWOT_ACC_SMST_20110913.nc").isel(time=0)[{"Eta", "XC", "YC", "DXV", "DYU"}]

In [ ]:
ds_tmp_coarse